# Recipe Generator — Notebook Monolitik (Tugas Besar IF5153)

Notebook ini menggabungkan **seluruh kode non-UI** repositori ke dalam satu file yang bisa
dibaca dan dijalankan end-to-end: preprocessing, ketiga pipeline, skrip data & pelatihan,
harness evaluasi/eksperimen, dan test suite. UI Streamlit (`app/`) tidak ikut digabung —
ia hanya mengimpor dari modul-modul yang sekarang didefinisikan di notebook ini.

**Pemetaan ke source** — setiap section merangkai file asli secara *verbatim* (hanya
import lintas-modul, guard CLI, dan `Path(__file__)` yang disesuaikan):

| Section | Sumber |
|---|---|
| 1. Preprocessing | `src/preprocessing/` |
| 2. Pipeline 1 | `src/pipeline1/` |
| 3. Pipeline 2 | `src/pipeline2/` |
| 4. Pipeline 3 | `src/pipeline3/` |
| 5. Skrip data & pelatihan | `scripts/` |
| 6. Evaluasi & eksperimen | `evaluation/` |
| 7. Test suite | `tests/` |
| 8. Demo end-to-end | — |

## 0. Setup — imports, konstanta, dan flag

Semua import eksternal dari seluruh modul di-hoist ke sini (verbatim, dedup). Konstanta
`ROOT`, `REPO_ROOT`, dan `SAMPLE_CSV` berasal dari definisi identik di berbagai file.


In [1]:
# Stdlib (hoisted dari seluruh modul)
from collections import Counter
from collections import defaultdict
from dataclasses import dataclass
from dataclasses import dataclass, field
from dataclasses import replace
from fractions import Fraction
from functools import lru_cache
from pathlib import Path
from typing import Any
from typing import Any, Iterable, Mapping, Sequence
from typing import Any, Mapping
from typing import Any, Mapping, Sequence
from typing import Iterable
from typing import Iterable, Mapping
import ast
import csv
import difflib
import io
import json
import math
import pickle
import random
import re
import statistics
import sys
import time
import unittest
import urllib.request
import zipfile

# Third-party
from scipy.sparse import csr_matrix
from scipy.sparse import csr_matrix, diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize as sk_normalize
import numpy as np
import pandas as pd
import pytest
import tempfile  # pengganti fixture pytest tmp_path


In [2]:
# Cari root repositori dari cwd (kernel Jupyter menjalankan notebook dari dir notebook-nya)
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), "root repositori tidak ditemukan dari cwd"
REPO_ROOT = ROOT                      # alias gaya penamaan scripts/tests
SAMPLE_CSV = ROOT / "data" / "samples" / "recipes_sample.csv"
print(f"repo root: {ROOT}")

repo root: /Users/iweng/Documents/GitHub/Tugas-Besar-IF5153-Recipe-Generator


In [3]:
# ---- Flag cold path / evaluasi -------------------------------------------
RUN_DOWNLOAD_DATA = False     # scripts/download_raw_data.py    (USDA + sampel RecipeNLG)
RUN_BUILD_REFERENCE = False   # scripts/build_calorie_reference.py
RUN_BUILD_BASELINE = False    # scripts/build_calorie_baseline.py
RUN_BUILD_CORPUS = False      # src/preprocessing/build_corpus.py (RecipeNLG -> parquet)
RUN_BUILD_QUERIES = False     # scripts/build_synthetic_queries.py
RUN_CHECK_COVERAGE = False    # scripts/check_ingredient_coverage.py
RUN_GENERATE_NER_DATA = False # scripts/generate_ner_training_data.py
RUN_TRAIN_NER = False         # scripts/train_ner.py  (DistilBERT, puluhan menit di MPS/CPU)
RUN_FIT_TFIDF = False         # src/pipeline2/vectorizer.py main
RUN_ENCODE_DENSE = False      # scripts/encode_dense_corpus.py (sentence-transformers)
RUN_SIMULATE = False          # scripts/simulate_pipeline2.py
RUN_EVALUATIONS = True        # harness evaluasi section 6 (read-only, hitungan menit)

In [4]:
# ---- Warm path: cek ketersediaan artifact --------------------------------
_expected = {
    "artifacts/ingredient_vocabulary.pkl":  "vocab Pipeline 1 (wajib untuk test & demo)",
    "data/processed/recipes.parquet":       "korpus terproses (wajib untuk retrieval)",
    "artifacts/tfidf_vectorizer.pkl":       "vektor TF-IDF (Pipeline 2)",
    "artifacts/recipe_vectors.pkl":         "matriks TF-IDF resep (Pipeline 2)",
    "artifacts/dense_vectors.npy":          "dense retrieval (opsional)",
    "artifacts/ner_model/config.json":      "model NER terlatih (opsional)",
}
for rel, note in _expected.items():
    p = ROOT / rel
    print(f"{'OK  ' if p.exists() else 'MISS'} {rel:<42} {note}")

OK   artifacts/ingredient_vocabulary.pkl        vocab Pipeline 1 (wajib untuk test & demo)
OK   data/processed/recipes.parquet             korpus terproses (wajib untuk retrieval)
OK   artifacts/tfidf_vectorizer.pkl             vektor TF-IDF (Pipeline 2)
OK   artifacts/recipe_vectors.pkl               matriks TF-IDF resep (Pipeline 2)
OK   artifacts/dense_vectors.npy                dense retrieval (opsional)
OK   artifacts/ner_model/config.json            model NER terlatih (opsional)


## 1. Preprocessing — `src/preprocessing/`

Lapisan paling bawah: normalisasi ingredient, parsing kalimat kuantitas, estimasi kalori,
dan pembangunan korpus terproses dari RecipeNLG.


**`src/preprocessing/ingredient_normalizer.py`**

Canonical ingredient normalization shared by every pipeline.  Two collaborators live here on purpose:  * ``Vocabulary`` — the closed query-time vocabulary used by Pipeline 1. Only ingredients with a calorie treatment in ``data/processed/`` plus the curated alias table can be recognised; longest surface phrase wins. * ``IngredientNormalizer`` — the offline normalizer used by corpus preprocessing and Pipeline 2 (alias resolution, suffix-trim scanning, recipe vocabulary).  Both canonicalize ingredient strings through ``normalize_ingredient`` (fold + head-noun singularization). ``normalize_text`` stays a pure query-time fold and must not singularize: it feeds whole user queries, including trailing exclusion clauses such as "no cheese" that would corrupt under naive stemming. ``~`` is kept there because the calorie constraint grammar reads ``~500`` as an approximate target. Rules mirror ``scripts/ingredient_tables.py``; keep the two in sync through ``data/processed/ingredient_aliases.csv``.


In [ ]:
from src.preprocessing.ingredient_normalizer import *


**`src/preprocessing/ingredient_parser.py`**


In [ ]:
from src.preprocessing.ingredient_parser import *


**`src/preprocessing/calorie_estimator.py`**

Recipe calorie estimator (PRD Sections 14-16) — the single one in the repo.  Merges the former ``CalorieMapper`` (dict API used by corpus preprocessing) and ``RecipeCalorieEstimator`` (typed API used by Pipeline 3, evaluation, UI) into one engine over the committed tables in ``data/processed/``:  - ``calorie_baseline.csv``         (deterministic canonical kcal/100g + fdc_id) - ``supplement_calories.csv``      (fallback kcal/100g) - ``zero_calorie_ingredients.csv`` (zero/negligible calorie items) - ``calorie_reference.csv``        (7,793 USDA SR Legacy foods for fallback matching) - ``unit_grams_reference.csv``     (per-fdc_id household unit -> grams) - ``unit_grams_fallback.csv``      (generic unit -> grams)  Lookup priority: zero -> baseline -> reference -> supplement -> fuzzy. Reference rows collapse to the median kcal per base ingredient (deterministic). Values returned are *Estimated Calories*, not authoritative nutritional labels (PRD Risk 2).


In [ ]:
from src.preprocessing.calorie_estimator import *


**`src/preprocessing/build_corpus.py`**


In [ ]:
from src.preprocessing.build_corpus import *

def run_build_corpus(corpus=DEFAULT_CORPUS_CSV, output=DEFAULT_OUTPUT_PARQUET, vocab=DEFAULT_VOCAB_PATH, limit=None) -> None:
    df = build_processed_corpus(
        corpus_path=corpus,
        output_path=output,
        vocab_path=vocab,
        limit=limit,
    )
    reliable_ratio = float((df["calorie_coverage"] >= 0.7).mean()) if len(df) else 0.0
    print(f"processed_recipes={len(df)} reliable_calorie_ratio={reliable_ratio:.4f}")


In [9]:
if RUN_BUILD_CORPUS:
    run_build_corpus()          # RecipeNLG penuh -> data/processed/recipes.parquet
    # run_build_corpus(limit=10000)  # korpus kecil untuk eksperimen cepat

## 2. Pipeline 1 — Pemahaman Query & Ekstraksi Ingredient (`src/pipeline1/`)

Query bebas -> `StructuredQuery` (ingredient tersedia, eksklusi, target kalori).
Ekstraksi ingredient punya 4 pendekatan *pluggable*: rule-based, surface-exact, fuzzy
(difflib / char n-gram), dan NER.


**`src/pipeline1/constraint_extractor.py`**

Pipeline 1 — calorie and exclusion constraint extraction (PRD §12.5–12.6).  Both extractors work on text that has already gone through ``normalize_text`` (lowercase, punctuation collapsed to spaces, apostrophes removed), so ``don't want`` arrives as ``dont want``.


In [ ]:
from src.pipeline1.constraint_extractor import *


**`src/pipeline1/ingredient_extractor.py`**

Pipeline 1 — ingredient mention extraction over the closed vocabulary.


In [ ]:
from src.pipeline1.ingredient_extractor import *


**`src/pipeline1/query_parser.py`**

Pipeline 1 — Query & Constraint Understanding (PRD §12).  raw query -> normalize -> exclusion/calorie constraints -> ingredient scan -> Structured Query  The output matches the PRD contract exactly:  { "available_ingredients": ["chicken", ...], "excluded_ingredients": ["cheese", ...], "calorie_target": 500 | None, "calorie_min": None, "calorie_max": None }  An ingredient named in an exclusion clause is never reported as available, even when it is also listed earlier ("I have butter... without butter").


In [ ]:
from src.pipeline1.query_parser import *


**`src/pipeline1/approaches/base.py`**

Approach contract and evidence records for Pipeline 1 matching.


In [ ]:
from src.pipeline1.approaches.base import *


**`src/pipeline1/approaches/rule.py`**

Vocabulary-scanning approaches: production rule matcher and the exact-surface ablation.


In [ ]:
from src.pipeline1.approaches.rule import *


**`src/pipeline1/approaches/fuzzy.py`**

Fuzzy-fallback approaches: production exact matches first, then similarity recovery.  Moved here from ``evaluation/experiment.py`` so the CLI harness, the tests, and the UI share one implementation. Both variants keep the production exact-match spans untouched and only try to rescue uncovered tokens.


In [ ]:
from src.pipeline1.approaches.fuzzy import *


**`src/pipeline1/approaches/ner.py`**

BERT-based token-classification approach for Pipeline 1 ingredient extraction.  Fine-tuned DistilBERT trained on synthetic pantry queries (see ``scripts/generate_ner_training_data.py`` and ``scripts/train_ner.py``). The model sees the same masked matching text the rule-based approaches see: calorie and exclusion spans are already blanked by the query parser.  Transformers/torch are imported lazily so the lexical approaches keep working without the deep-learning stack installed.


In [ ]:
from src.pipeline1.approaches.ner import *


**`src/pipeline1/approaches/runner.py`**

Trace-aware Pipeline 1 runner: same composition as ``query_parser.parse_query``, recording every intermediate step for the UI.


In [ ]:
from src.pipeline1.approaches.runner import *


**`src/pipeline1/approaches/__init__.py`**

Pipeline 1 — pluggable ingredient-matching approaches.  Every approach answers the same contract so the query parser, the experiment harness, and the Streamlit UI can swap them freely:  * ``extract(text, vocab) -> list[str]`` — canonical ingredients, first-mention order (drop-in for ``parse_query``'s ``matcher`` argument); * ``trace(text, vocab) -> ApproachTrace`` — the same result plus the intermediate evidence (matched spans, provenance, scores) the UI renders.  The matching context is the *masked* query text produced by ``query_parser.parse_query``: calorie and exclusion spans are already blanked, so approaches never see constraint clauses.


In [ ]:
from src.pipeline1.approaches import *


## 3. Pipeline 2 — Retrieval Kandidat (`src/pipeline2/`)

TF-IDF (dengan down-weighting pantry staple) dan dense retrieval (sentence-transformers)
menghasilkan kandidat ber-skema identik untuk dikonsumsi reranker Pipeline 3.


**`src/pipeline2/vectorizer.py`**


In [ ]:
from src.pipeline2.vectorizer import *

def run_fit_tfidf(corpus=DEFAULT_CORPUS_PARQUET, vectorizer_out=DEFAULT_VECTORIZER_PATH, vectors_out=DEFAULT_VECTORS_PATH, max_features=0, ngram_max=1, no_sublinear_tf=False, no_staple_downweight=False) -> None:
    max_features = None if max_features <= 0 else max_features
    _, vectors = fit_corpus_tfidf(
        corpus_parquet=corpus,
        vectorizer_path=vectorizer_out,
        vectors_path=vectors_out,
        max_features=max_features,
        ngram_range=(1, ngram_max),
        sublinear_tf=not no_sublinear_tf,
        use_staple_downweight=not no_staple_downweight,
    )
    print(f"fitted_shape={vectors.shape} nnz={vectors.nnz}")


In [20]:
if RUN_FIT_TFIDF:
    run_fit_tfidf()             # -> artifacts/tfidf_vectorizer.pkl + recipe_vectors.pkl

**`src/pipeline2/retriever.py`**


In [ ]:
from src.pipeline2.retriever import *


**`src/pipeline2/approaches/__init__.py`**

Pipeline 2 — pluggable retrieval approaches.  Both approaches answer one contract so the query parser output, the reranker, and the UI can swap them freely:  * ``retrieve(structured_query, top_n) -> list[dict]`` — candidate dicts with the exact schema ``pipeline3.models.CandidateRecipe.from_dict`` consumes (``recipe_id``, ``title``, ``ingredients``, ``canonical_ingredients``, ``directions``, ``total_calories``, ``calorie_coverage``, ``tfidf_score``); * ``trace(structured_query, top_n) -> RetrievalTrace`` — query terms, weights, and per-candidate scores for the UI.


In [ ]:
from src.pipeline2.approaches import (
    DEFAULT_DENSE_META_PATH,
    DEFAULT_DENSE_VECTORS_PATH,
    DEFAULT_MODEL_NAME,
    DenseApproach,
    RetrievalApproach,
    RetrievalTrace,
    TfidfApproach,
    available_approaches as available_retrieval_approaches,
    get_approach as get_retrieval_approach,
)


## 4. Pipeline 3 — Pantry-Aware Multi-Criteria Reranking (`src/pipeline3/`)

Kandidat Pipeline 2 dinilai ulang: coverage pantry, deteksi eksklusi, skor kalori,
digabung `MultiCriteriaReranker` menjadi rekomendasi akhir + kartu rekomendasi.


**`src/pipeline3/models.py`**

Data models and configuration schemas for Pipeline 3 (Multi-Criteria Reranking).  These dataclasses define the clean integration contract between: - Pipeline 1 (Query & Constraint Understanding) -> ``StructuredQuery`` - Pipeline 2 (Candidate Recipe Retrieval) -> ``CandidateRecipe`` - Pipeline 3 (Multi-Criteria Recipe Reranking) -> ``RecommendationResult``


In [ ]:
from src.pipeline3.models import *


**`src/pipeline3/coverage.py`**

Ingredient coverage, missing ingredient penalty, and pantry staple weights (PRD §§19-21).


In [ ]:
from src.pipeline3.coverage import *


**`src/pipeline3/exclusion.py`**

Excluded ingredient detection and filtering (PRD Section 23).  Handles both exact canonical matches and generic-category exclusions such as "no cheese" (matching cheddar cheese, mozzarella cheese, parmesan, cream cheese, etc.) or "no nuts" (matching almond, walnut, pecan, peanut, mixed nuts) while guarding common false-positive compound names (e.g., "peanut butter" is not dairy butter; "cream of tartar" is not dairy cream; "nutmeg" is not a nut).  Note (PRD Section 6): Excluded ingredients are treated as user preference constraints, not medical allergy guarantees.


In [ ]:
from src.pipeline3.exclusion import *


**`src/pipeline3/calorie_score.py`**

Calorie compatibility scoring and hard-bound checking (PRD Section 22).


In [ ]:
from src.pipeline3.calorie_score import *


**`src/pipeline3/reranker.py`**

Multi-criteria recipe reranker with constraint filtering and fallback (PRD §§24-25 & Risk 6).


In [ ]:
from src.pipeline3.reranker import *


**`src/pipeline3/recommender.py`**

Explainable recommendation entry point, formatting, and serialization (PRD §§26-27).


In [ ]:
from src.pipeline3.recommender import *


## 5. Skrip Data & Pelatihan — `scripts/`

Seluruh skrip CLI menjadi fungsi `run_*` (argparse -> parameter dengan default yang sama),
dipanggil lewat flag `RUN_*` di cell guard. Tidak ada yang jalan kecuali flag-nya dinyalakan.


**`scripts/ingredient_tables.py`**

Dataset tooling normalization — the single source lives in src/preprocessing.  Re-exports the same rules Pipeline 1-3 use (fold + head-noun singularization, alias resolution) so dataset tooling can no longer drift from pipeline code.


In [29]:
PROCESSED = ROOT / "data" / "processed"

# alias dari src.preprocessing.ingredient_normalizer (sudah didefinisikan di section 1)
Resolver = IngredientNormalizer
normalize = normalize_ingredient
read_rows = read_csv_rows
# singularize sudah tersedia di namespace


**`scripts/download_raw_data.py`**

Fetch every external dataset YummyTummy needs (stdlib only).  Dataset inventory and rationale: data/README.md.  python scripts/download_raw_data.py --calories          # USDA FoodData Central (~7 MB) python scripts/download_raw_data.py --recipes sample    # ~7 MB random sample python scripts/download_raw_data.py --recipes full      # 2.3 GB full RecipeNLG corpus python scripts/download_raw_data.py --all               # calories + sample


In [30]:
RAW_RECIPES = ROOT / "data" / "raw" / "recipenlg"
RAW_CALORIES = ROOT / "data" / "raw" / "calories"
SAMPLES = ROOT / "data" / "samples"

# RecipeNLG original columns: title, ingredients, directions, link, source, NER.
# HuggingFace mirrors are used because recipenlg.cs.put.poznan.pl has been
# unreachable since 2025 and Kaggle needs an authenticated API token.
RECIPE_MIRRORS = [
    "https://huggingface.co/datasets/Mahimas/recipenlg/resolve/main/RecipeNLG.csv",
    "https://huggingface.co/datasets/SandhyaKilari/RecipeNLG_dataset/resolve/main/RecipeNLG_dataset.csv",
]
RECIPE_COLUMNS = ["title", "ingredients", "directions", "link", "source", "NER"]

# USDA FoodData Central, public domain (CC0).
# SR Legacy only: 7,793 foods, the last complete generic-ingredient release.
# Foundation Foods switched to a per-sample schema without food_nutrient.csv.
FDC_FILES = {
    "sr_legacy": "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_sr_legacy_food_csv_2018-04.zip",
}

# The corpus is ordered by source, so a prefix sample would be single-source.
# With the full file on disk we take a uniform random sample; without it we fall
# back to equal-size byte windows spread across the whole file.
SAMPLE_ROWS = 10_000
SAMPLE_SEED = 42
SAMPLE_CHUNKS = 12
SAMPLE_CHUNK_BYTES = 800_000

def _download(url: str, dest: Path) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": "yummytummy-dataset-prep/1.0"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        total = int(resp.headers.get("Content-Length") or 0)
        seen = 0
        t0 = time.time()
        with dest.open("wb") as fh:
            while chunk := resp.read(1 << 20):
                fh.write(chunk)
                seen += len(chunk)
                if total:
                    print(f"\r  {dest.name}: {seen / total:6.1%} ({seen >> 20} / {total >> 20} MB)", end="")
                else:
                    print(f"\r  {dest.name}: {seen >> 20} MB", end="")
        print(f"\r  {dest.name}: {seen >> 20} MB in {time.time() - t0:.0f}s")

def _range(url: str, start: int, end: int) -> bytes:
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "yummytummy-dataset-prep/1.0",
            "Range": f"bytes={start}-{end}",
        },
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return resp.read()

def _corpus_size(url: str) -> int:
    req = urllib.request.Request(url, headers={"User-Agent": "yummytummy-dataset-prep/1.0", "Range": "bytes=0-0"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        content_range = resp.headers.get("Content-Range", "")
        if "/" in content_range:
            return int(content_range.rsplit("/", 1)[1])
        return int(resp.headers["Content-Length"])

def _parse_recipe_row(row: list[str], keep: list[int]) -> list[str] | None:
    """Return the six RecipeNLG columns, or None for a truncated/corrupt row."""
    if len(row) <= max(keep):
        return None
    try:
        ast.literal_eval(row[keep[1]])  # ingredients
        ast.literal_eval(row[keep[-1]])  # NER
    except (ValueError, SyntaxError, TypeError):
        return None
    return [row[i] for i in keep]

def _normalize_recipes(chunks: list[bytes], dest: Path) -> int:
    """Drop the unnamed pandas index column, keep the six RecipeNLG columns."""
    header: list[str] | None = None
    keep: list[int] = []
    written = 0
    with dest.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(RECIPE_COLUMNS)
        for index, raw in enumerate(chunks):
            rows = list(csv.reader(io.StringIO(raw.decode("utf-8", errors="ignore"), newline="")))
            if header is None:
                header = rows.pop(0)
                keep = [header.index(c) for c in RECIPE_COLUMNS]
            else:
                rows.pop(0)  # starts inside a previous record
            if not raw.endswith(b"\n"):
                rows.pop()  # last record is cut off
            for row in rows:
                record = _parse_recipe_row(row, keep)
                if record:
                    writer.writerow(record)
                    written += 1
    return written

def _write_sample(records: list[list[str]], dest: Path) -> int:
    with dest.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(RECIPE_COLUMNS)
        writer.writerows(records)
    return len(records)

def _sample_local(path: Path, count: int, seed: int) -> list[list[str]]:
    """Reservoir sample `count` valid records in corpus order (one pass over the file)."""
    rng = random.Random(seed)
    picked: list[tuple[int, list[str]]] = []
    keep: list[int] = []
    seen = 0
    with path.open(encoding="utf-8", newline="") as fh:
        reader = csv.reader(fh)
        header = next(reader)
        keep = [header.index(c) for c in RECIPE_COLUMNS]
        for row in reader:
            if len(row) <= max(keep):  # truncated row
                continue
            if len(picked) < count:
                picked.append((seen, row))
            else:
                slot = rng.randrange(seen + 1)
                if slot < count:
                    picked[slot] = (seen, row)
            seen += 1
    picked.sort(key=lambda item: item[0])
    records = []
    for _, row in picked:
        record = _parse_recipe_row(row, keep)
        if record:
            records.append(record)
    print(f"[info] {seen} corpus rows scanned, {len(records)} sampled")
    return records

def fetch_recipe_sample(force: bool = False) -> Path:
    dest = SAMPLES / "recipes_sample.csv"
    if dest.exists() and not force:
        print(f"[skip] {dest.relative_to(ROOT)} exists")
        return dest
    SAMPLES.mkdir(parents=True, exist_ok=True)
    full = RAW_RECIPES / "RecipeNLG.csv"
    if full.exists():
        print(f"[recipes] sampling {SAMPLE_ROWS} rows from local {full.relative_to(ROOT)}")
        n = _write_sample(_sample_local(full, SAMPLE_ROWS, SAMPLE_SEED), dest)
        print(f"[ok] {dest.relative_to(ROOT)}: {n} recipes")
        return dest
    last_err: Exception | None = None
    for url in RECIPE_MIRRORS:
        try:
            size = _corpus_size(url)
            print(f"[recipes] sampling {SAMPLE_CHUNKS} windows of {SAMPLE_CHUNK_BYTES >> 10} KB from {size / 1e9:.2f} GB")
            chunks = []
            for i in range(SAMPLE_CHUNKS):
                start = i * size // SAMPLE_CHUNKS
                blob = _range(url, start, min(start + SAMPLE_CHUNK_BYTES - 1, size - 1))
                if len(blob) < SAMPLE_CHUNK_BYTES // 2:
                    raise RuntimeError(f"short range response at byte {start}")
                chunks.append(blob)
            n = _normalize_recipes(chunks, dest)  # network fallback
            print(f"[ok] {dest.relative_to(ROOT)}: {n} recipes")
            return dest
        except Exception as err:  # noqa: BLE001 - try the next mirror
            last_err = err
            print(f"[warn] mirror failed: {err}")
    raise SystemExit(f"all RecipeNLG mirrors failed: {last_err}")

def fetch_recipe_full(force: bool = False) -> Path:
    dest = RAW_RECIPES / "RecipeNLG.csv"
    if dest.exists() and not force:
        print(f"[skip] {dest.relative_to(ROOT)} exists ({dest.stat().st_size >> 20} MB)")
        return dest
    RAW_RECIPES.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".csv.part")
    last_err: Exception | None = None
    for url in RECIPE_MIRRORS:
        try:
            print(f"[recipes] downloading 2.3 GB corpus from {url}")
            _download(url, tmp)
            tmp.replace(dest)
            with dest.open(encoding="utf-8", newline="") as fh:
                header = next(csv.reader(fh))
            missing = [c for c in RECIPE_COLUMNS if c not in header]
            if missing:
                raise SystemExit(f"{dest.name}: missing columns {missing}, got {header}")
            print(f"[ok] {dest.relative_to(ROOT)} ({dest.stat().st_size >> 20} MB)")
            return dest
        except SystemExit:
            raise
        except Exception as err:  # noqa: BLE001
            last_err = err
            print(f"[warn] mirror failed: {err}")
    tmp.unlink(missing_ok=True)
    raise SystemExit(f"all RecipeNLG mirrors failed: {last_err}")

def fetch_calories(force: bool = False) -> None:
    RAW_CALORIES.mkdir(parents=True, exist_ok=True)
    for name, url in FDC_FILES.items():
        target = RAW_CALORIES / name
        if target.is_dir() and any(target.glob("food.csv")) and not force:
            print(f"[skip] data/raw/calories/{name} exists")
            continue
        print(f"[calories] {name}")
        archive = RAW_CALORIES / f"{name}.zip"
        _download(url, archive)
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                rel = Path(member).parts[1:]  # strip the archive's top folder
                if not rel or member.endswith("/"):
                    continue
                out = target.joinpath(*rel)
                out.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, out.open("wb") as dst:
                    dst.write(src.read())
        archive.unlink()
        print(f"[ok] data/raw/calories/{name}")

def run_download_raw_data(recipes='none', calories=False, everything=False, force=False) -> None:

    if everything:
        calories, recipes = True, "sample"
    if calories:
        fetch_calories(force)
    if recipes == "sample":
        fetch_recipe_sample(force)
    elif recipes == "full":
        fetch_recipe_full(force)
    if not calories and recipes == "none":
        print("[notebook] tidak ada aksi: berikan parameter yang diinginkan")


In [31]:
if RUN_DOWNLOAD_DATA:
    run_download_raw_data(everything=True)      # tabel USDA + sampel 10k resep
    # run_download_raw_data(recipes="full")     # RecipeNLG penuh (~2.2M resep)

**`scripts/build_calorie_reference.py`**

Build the calorie reference tables from USDA FoodData Central CSVs (stdlib only).  Input  : data/raw/calories/sr_legacy/     (download_raw_data.py --calories) Output : data/processed/calorie_reference.csv      one row per USDA food data/processed/unit_grams_reference.csv   (fdc_id, unit) -> grams  Every value is kcal per 100 g edible portion, taken from nutrient 1008 (Energy) with the Atwater factors (2047/2048) as fallback. FoodData Central is public domain (CC0); see data/README.md.


In [32]:
RAW = ROOT / "data" / "raw" / "calories"
REFERENCE_OUT_DIR = ROOT / "data" / "processed"

# Same plural folding the pipelines use (stdlib-only module, keeps USDA keys aligned).

ENERGY_IDS = ("1008", "2047", "2048")  # Energy kcal, Atwater general, Atwater specific

# USDA `modifier` values -> unit tokens the ingredient parser emits.
UNIT_PATTERNS = [
    (r"^cup", "cup"),
    (r"^(tbsp|tablespoon)", "tbsp"),
    (r"^(tsp|teaspoon)", "tsp"),
    (r"^(fl oz|fluid ounce)", "fl_oz"),
    (r"^(oz|ounce)", "oz"),
    (r"^(lb|pound)", "lb"),
    (r"^(g|gram)", "g"),
    (r"^(ml|milliliter)", "ml"),
    (r"^slice", "slice"),
    (r"^clove", "clove"),
    (r"^(can|jar|package|pkg|stick|bar|bunch|head|stalk|piece|pinch|dash|drop|sprig)", None),
]

def canonical_unit(modifier: str) -> str | None:
    m = modifier.strip().lower()
    for pattern, unit in UNIT_PATTERNS:
        if re.match(pattern, m):
            return unit or m.split(",")[0].split()[0]
    return None

def base_ingredient(description: str) -> str:
    """'Butter, salted' -> 'butter'; 'Carrots, raw' -> 'carrot'."""
    head = description.split(",")[0]
    head = re.sub(r"\(.*?\)", " ", head)
    head = re.sub(r"[^a-z0-9 ]+", " ", head.lower())
    head = " ".join(head.split())
    if not head:
        return head
    tokens = head.split(" ")
    tokens[-1] = singularize(tokens[-1])
    return " ".join(tokens)

def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))

def load_energy() -> tuple[dict[str, tuple[float, str]], dict[str, dict]]:
    """fdc_id -> (kcal per 100 g, nutrient id used)."""
    rows: dict[str, list[tuple[int, float, str]]] = defaultdict(list)
    for source in sorted(RAW.iterdir()):
        if not (source / "food_nutrient.csv").exists():
            continue
        for row in read_csv(source / "food_nutrient.csv"):
            nid = row["nutrient_id"]
            if nid not in ENERGY_IDS or not row["amount"]:
                continue
            try:
                amount = float(row["amount"])
            except ValueError:
                continue
            rows[row["fdc_id"]].append((ENERGY_IDS.index(nid), amount, nid))
    best = {}
    for fdc_id, candidates in rows.items():
        candidates.sort()
        rank, amount, nid = candidates[0]
        best[fdc_id] = (amount, nid)
    return best

def load_foods() -> dict[str, dict]:
    foods = {}
    for source in sorted(RAW.iterdir()):
        if not (source / "food.csv").exists():
            continue
        for row in read_csv(source / "food.csv"):
            foods[row["fdc_id"]] = row
    return foods

def build_reference(foods: dict[str, dict], energy: dict) -> list[dict]:
    reference = []
    for fdc_id, row in foods.items():
        if fdc_id not in energy:
            continue
        kcal, nid = energy[fdc_id]
        reference.append(
            {
                "fdc_id": fdc_id,
                "description": row["description"].strip(),
                "base_ingredient": base_ingredient(row["description"]),
                "kcal_per_100g": round(kcal, 2),
                "energy_nutrient_id": nid,
                "data_type": row["data_type"],
                "food_category_id": row["food_category_id"],
            }
        )
    reference.sort(key=lambda r: (r["base_ingredient"], r["description"]))
    return reference

def build_unit_grams(reference: list[dict]) -> list[dict]:
    base_by_fdc = {row["fdc_id"]: row["base_ingredient"] for row in reference}
    buckets: dict[tuple[str, str, str], list[float]] = defaultdict(list)
    for source in sorted(RAW.iterdir()):
        path = source / "food_portion.csv"
        if not path.exists():
            continue
        for row in read_csv(path):
            ingredient = base_by_fdc.get(row["fdc_id"])
            unit = canonical_unit(row["modifier"] or row["portion_description"] or "")
            if not ingredient or not unit or not row["gram_weight"]:
                continue
            try:
                amount = float(row["amount"] or 1)
                grams = float(row["gram_weight"])
            except ValueError:
                continue
            if amount <= 0 or grams <= 0 or grams / amount > 5000:
                continue
            buckets[(row["fdc_id"], ingredient, unit)].append(grams / amount)
    out = []
    for (fdc_id, ingredient, unit), values in sorted(buckets.items(), key=lambda kv: (kv[0][1], kv[0][2], kv[0][0])):
        out.append(
            {
                "fdc_id": fdc_id,
                "ingredient": ingredient,
                "unit": unit,
                "grams_per_unit": round(statistics.median(values), 2),
                "n_portions": len(values),
                "source": "USDA FDC food_portion",
            }
        )
    return out

def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"[ok] {path.relative_to(ROOT)}: {len(rows)} rows")

def run_build_calorie_reference() -> None:
    if not RAW.exists() or not any(RAW.iterdir()):
        raise SystemExit("data/raw/calories is empty: run download_raw_data.py --calories")

    foods = load_foods()
    energy = load_energy()
    reference = build_reference(foods, energy)
    covered = len(reference) / max(len(foods), 1)
    print(f"[info] {len(foods)} USDA foods, {len(reference)} with an energy value ({covered:.1%})")

    write_csv(REFERENCE_OUT_DIR / "calorie_reference.csv", reference)
    write_csv(REFERENCE_OUT_DIR / "unit_grams_reference.csv", build_unit_grams(reference))


In [33]:
if RUN_BUILD_REFERENCE:
    run_build_calorie_reference()

**`scripts/build_calorie_baseline.py`**

Build the curated ingredient -> kcal/100 g baseline used before fuzzy matching.  `calorie_reference.csv` is the raw USDA search space (7,793 foods, many variants per ingredient). This script pins one deliberate USDA description per canonical ingredient so the calorie mapper has a deterministic first hit; anything not listed here falls through to exact -> normalized -> fuzzy -> supplement.  Input  : data/processed/calorie_reference.csv Output : data/processed/calorie_baseline.csv


In [34]:
REFERENCE = ROOT / "data" / "processed" / "calorie_reference.csv"
BASELINE_OUT = ROOT / "data" / "processed" / "calorie_baseline.csv"

# canonical ingredient -> tokens that must all appear in the USDA description.
# The first non-processed, shortest match wins, so keep tokens discriminating.
MAP: dict[str, tuple[str, ...]] = {
    # pantry / baking
    "salt": ("salt, table",),
    "sugar": ("sugars, granulated",),
    "brown sugar": ("sugars, brown",),
    "powdered sugar": ("sugars, powdered",),
    "flour": ("wheat flour", "all-purpose"),
    "whole wheat flour": ("wheat flour", "whole-grain"),
    "cornstarch": ("cornstarch",),
    "baking powder": ("leavening agents, baking powder", "double-acting"),
    "baking soda": ("leavening agents, baking soda",),
    "yeast": ("yeast, baker's",),
    "cornmeal": ("cornmeal, whole-grain, yellow",),
    "oat": ("cereals, oats", "regular and quick", "dry"),
    "bread crumb": ("bread, crumbs", "plain"),
    "cracker crumb": ("crackers, saltines",),
    "gelatin": ("gelatins, dry powder",),
    "cocoa": ("cocoa, dry powder, unsweetened",),
    "chocolate chip": ("candies, semisweet chocolate",),
    "vanilla": ("vanilla extract",),
    "honey": ("honey",),
    "molasses": ("molasses",),
    "maple syrup": ("syrup, maple",),
    "peanut butter": ("peanut butter, smooth",),
    # fats and dairy
    "butter": ("butter, salted",),
    "margarine": ("margarine, regular",),
    "shortening": ("shortening, household",),
    "oil": ("oil, soybean, salad or cooking",),
    "olive oil": ("oil, olive",),
    "vegetable oil": ("oil, soybean, salad or cooking",),
    "milk": ("milk, whole, 3.25%",),
    "evaporated milk": ("milk, evaporated, 2%",),
    "condensed milk": ("milk, canned, condensed, sweetened",),
    "buttermilk": ("milk, buttermilk, fluid, cultured, lowfat",),
    "cream": ("cream, fluid, heavy whipping",),
    "heavy cream": ("cream, fluid, heavy whipping",),
    "half and half": ("cream, fluid, half and half",),
    "sour cream": ("cream, sour, cultured",),
    "yogurt": ("yogurt, plain, whole milk",),
    "cream cheese": ("cheese, cream",),
    "cheddar cheese": ("cheese, cheddar",),
    "parmesan cheese": ("cheese, parmesan, hard",),
    "mozzarella cheese": ("cheese, mozzarella, whole milk",),
    "swiss cheese": ("cheese, swiss",),
    "ricotta cheese": ("cheese, ricotta, whole milk",),
    "cottage cheese": ("cheese, cottage, creamed", "curd"),
    "egg": ("egg, whole, raw, fresh",),
    "egg white": ("egg, white, raw, fresh",),
    "egg yolk": ("egg, yolk, raw, fresh",),
    # vegetables
    "onion": ("onions, raw",),
    "green onion": ("onions, spring or scallions",),
    "garlic": ("garlic, raw",),
    "potato": ("potatoes, flesh and skin, raw",),
    "sweet potato": ("sweet potato, raw",),
    "carrot": ("carrots, raw",),
    "celery": ("celery, raw",),
    "tomato": ("tomatoes, red, ripe, raw",),
    "tomato paste": ("tomato products, canned, paste",),
    "tomato sauce": ("tomato products, canned, sauce",),
    "green pepper": ("peppers, sweet, green, raw",),
    "red pepper": ("peppers, sweet, red, raw",),
    "bell pepper": ("peppers, sweet, green, raw",),
    "jalapeno": ("peppers, jalapeno, raw",),
    "mushroom": ("mushrooms, white, raw",),
    "spinach": ("spinach, raw",),
    "lettuce": ("lettuce, iceberg",),
    "cabbage": ("cabbage, raw",),
    "broccoli": ("broccoli, raw",),
    "cauliflower": ("cauliflower, raw",),
    "zucchini": ("squash, summer, zucchini", "raw"),
    "eggplant": ("eggplant, raw",),
    "arugula": ("arugula, raw",),
    "asparagus": ("asparagus, raw",),
    "lime": ("limes, raw",),
    "shallot": ("shallots, raw",),
    "mint": ("peppermint, fresh",),
    "caper": ("capers, canned",),
    "tarragon": ("spices, tarragon, dried",),
    "coffee": ("beverages, coffee, brewed, breakfast blend",),
    "mango": ("mangos, raw",),
    "raspberry": ("raspberries, raw",),
    "butternut squash": ("squash, winter, butternut, raw",),
    "summer squash": ("squash, summer, all varieties, raw",),
    "rhubarb": ("rhubarb, raw",),
    "hazelnut": ("nuts, hazelnuts or filberts",),
    "beet": ("beets, raw",),
    "pumpkin puree": ("pumpkin, canned",),
    "bean sprout": ("mung beans, mature seeds, sprouted, raw",),
    "couscous": ("couscous, dry",),
    "radish": ("radishes, raw",),
    "pork tenderloin": ("pork, fresh, loin, tenderloin", "cooked, broiled"),
    "beer": ("alcoholic beverage, beer, regular, all",),
    "sake": ("alcoholic beverage, rice (sake)",),
    "whiskey": ("alcoholic beverage, distilled, all", "80 proof"),
    "pretzel": ("snacks, pretzels, hard, plain",),
    "cucumber": ("cucumber, with peel, raw",),
    "green bean": ("beans, snap, green, raw",),
    "pea": ("peas, green, raw",),
    "corn": ("corn, sweet, yellow, raw",),
    "pumpkin": ("pumpkin, raw",),
    "avocado": ("avocados, raw, all commercial varieties",),
    "olive": ("olives, ripe, canned",),
    "ginger": ("ginger root, raw",),
    "pineapple": ("pineapple, raw, all varieties",),
    "peach": ("peaches, yellow, raw",),
    "cherry": ("cherries, sweet, raw",),
    "date": ("dates, medjool",),
    "tomato juice": ("tomato juice, canned",),
    "horseradish": ("horseradish, prepared",),
    "sauerkraut": ("sauerkraut, canned",),
    "applesauce": ("applesauce, canned, unsweetened",),
    "apple juice": ("apple juice, canned or bottled, unsweetened",),
    "potato chip": ("snacks, potato chips, plain, salted",),
    "mixed vegetable": ("vegetables, mixed, canned",),
    "cranberry sauce": ("cranberry sauce, canned, sweetened",),
    "red wine": ("alcoholic beverage, wine, table, red",),
    "chocolate cake": ("cake, chocolate, prepared from recipe without frosting",),
    "apple": ("apples, raw, with skin",),
    "banana": ("bananas, raw",),
    "orange": ("oranges, raw, all commercial varieties",),
    "lemon": ("lemons, raw, without peel",),
    "lemon juice": ("lemon juice, raw",),
    "lime juice": ("lime juice, raw",),
    "strawberry": ("strawberries, raw",),
    "blueberry": ("blueberries, raw",),
    "raisin": ("raisins, dark, seedless",),
    "coconut": ("nuts, coconut meat, raw",),
    # proteins
    "chicken": ("chicken, broilers or fryers, meat and skin, raw",),
    "chicken breast": ("chicken, broilers or fryers, breast", "raw"),
    "chicken thigh": ("chicken, broilers or fryers, thigh", "raw"),
    "ground beef": ("beef, ground, 80% lean meat / 20% fat", "raw"),
    "beef": ("beef, chuck, arm pot roast", "raw"),
    "pork": ("pork, fresh, loin, blade", "raw"),
    "bacon": ("pork, cured, bacon, unprepared",),
    "ham": ("ham, sliced, regular",),
    "sausage": ("pork sausage, link/patty, unprepared",),
    "turkey": ("turkey, whole, meat and skin, raw",),
    "shrimp": ("crustaceans, shrimp, raw",),
    "crab": ("crustaceans, crab, blue, raw",),
    "tuna": ("fish, tuna, light, canned in water", "drained"),
    "salmon": ("fish, salmon, atlantic, farmed, raw",),
    "cod": ("fish, cod, atlantic, raw",),
    "tilapia": ("fish, tilapia, raw",),
    "anchovy": ("fish, anchovy, european, raw",),
    "sardine": ("fish, sardine, atlantic, canned in oil",),
    "tofu": ("tofu, raw, firm",),
    "lentil": ("lentils, raw",),
    "chickpea": ("chickpeas", "mature seeds, raw"),
    "kidney bean": ("beans, kidney, all types", "raw"),
    "black bean": ("beans, black, mature seeds, raw",),
    "pinto bean": ("beans, pinto, mature seeds, raw",),
    # grains and pasta
    "rice": ("rice, white, long-grain, regular, raw",),
    "brown rice": ("rice, brown, long-grain, raw",),
    "pasta": ("pasta, dry, enriched",),
    "spaghetti": ("pasta, dry, enriched",),
    "macaroni": ("pasta, dry, enriched",),
    "noodle": ("noodles, egg, dry, enriched",),
    "quinoa": ("quinoa, uncooked",),
    "bread": ("bread, white, commercially prepared",),
    "tortilla": ("tortillas, ready-to-bake or -fry, flour",),
    "oatmeal": ("cereals, oats, regular and quick", "dry"),
    # nuts and seeds
    "almond": ("nuts, almonds",),
    "walnut": ("nuts, walnuts, english",),
    "pecan": ("nuts, pecans",),
    "cashew": ("nuts, cashew nuts, raw",),
    "peanut": ("peanuts, all types, raw",),
    "pistachio": ("nuts, pistachio nuts, raw",),
    "sunflower seed": ("seeds, sunflower seed kernels, dried",),
    "pumpkin seed": ("seeds, pumpkin and squash seed kernels, dried",),
    "sesame seed": ("seeds, sesame seeds, whole, dried",),
    "poppy seed": ("spices, poppy seed",),
    "flaxseed": ("seeds, flaxseed",),
    "chia seed": ("seeds, chia seeds, dried",),
    # condiments and seasoning
    "pepper": ("spices, pepper, black",),
    "salt and pepper": ("spices, pepper, black",),
    "cinnamon": ("spices, cinnamon, ground",),
    "nutmeg": ("spices, nutmeg, ground",),
    "paprika": ("spices, paprika",),
    "cumin": ("spices, cumin seed",),
    "oregano": ("spices, oregano, dried",),
    "thyme": ("spices, thyme, dried",),
    "rosemary": ("spices, rosemary, dried",),
    "basil": ("spices, basil, dried",),
    "parsley": ("parsley, fresh",),
    "cilantro": ("coriander (cilantro) leaves, raw",),
    "bay leaf": ("spices, bay leaf",),
    "chili powder": ("spices, chili powder",),
    "cayenne": ("spices, pepper, red or cayenne",),
    "garlic powder": ("spices, garlic powder",),
    "onion powder": ("spices, onion powder",),
    "mustard": ("mustard, prepared, yellow",),
    "mayonnaise": ("mayonnaise, regular",),
    "ketchup": ("catsup",),
    "vinegar": ("vinegar, distilled",),
    "soy sauce": ("soy sauce made from soy and wheat", "shoyu"),
    "worcestershire sauce": ("sauce, worcestershire",),
    "hot sauce": ("sauce, ready-to-serve, pepper or hot",),
    "salsa": ("sauce, salsa, ready-to-serve",),
    "chicken broth": ("soup, chicken broth, ready-to-serve",),
    "beef broth": ("soup, beef broth or bouillon canned, ready-to-serve",),
    "water": ("water, tap",),
}

EXCLUDE = (
    "meatless", "imitation", "substitute", "low-sodium", "low sodium", "low fat",
    "fat free", "reduced", "toasted", "sharp", "babyfood", "fast food", "restaurant",
)

PROCESSED_WORDS = (
    "dehydrated", "dried", "canned", "frozen", "cooked", "powder", "syrup",
    "sweetened", "juice", "concentrate", "fried", "babyfood", "restaurant",
    "fast food", "candies", "cereal", "snack", "mix", "prepared", "pasteurized",
)

def load_reference() -> list[dict]:
    with REFERENCE.open(encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))

def pick(reference: list[dict], tokens: tuple[str, ...]) -> dict | None:
    hits = [
        row
        for row in reference
        if all(token in row["description"].lower() for token in tokens)
        and not any(word in row["description"].lower() for word in EXCLUDE)
    ]
    if not hits:
        return None
    hits.sort(key=lambda r: (any(w in r["description"].lower() for w in PROCESSED_WORDS), len(r["description"])))
    return hits[0]

def run_build_calorie_baseline() -> None:
    if not REFERENCE.exists():
        raise SystemExit("missing data/processed/calorie_reference.csv: run build_calorie_reference.py")
    reference = load_reference()
    rows, unresolved = [], []
    for ingredient, tokens in MAP.items():
        match = pick(reference, tokens)
        if match is None:
            unresolved.append((ingredient, tokens))
            continue
        rows.append(
            {
                "ingredient": ingredient,
                "kcal_per_100g": match["kcal_per_100g"],
                "matched_description": match["description"],
                "fdc_id": match["fdc_id"],
                "source": "USDA FDC SR Legacy",
            }
        )
    rows.sort(key=lambda r: r["ingredient"])
    with BASELINE_OUT.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"[ok] {BASELINE_OUT.relative_to(ROOT)}: {len(rows)} ingredients")
    for ingredient, tokens in unresolved:
        print(f"[warn] no USDA match for {ingredient!r} ({' '.join(tokens)})", file=sys.stderr)


In [35]:
if RUN_BUILD_BASELINE:
    run_build_calorie_baseline()

**`scripts/build_synthetic_queries.py`**

Generate the Pipeline 2 evaluation set from the recipe corpus (PRD section 32).  For each sampled recipe, a random subset of its canonical ingredients becomes a simulated pantry. The source recipe is the relevance target: retrieval succeeds when it appears in the candidate list.  python scripts/build_synthetic_queries.py python scripts/build_synthetic_queries.py --corpus data/raw/recipenlg/RecipeNLG.csv --queries 500  Deterministic for a fixed --seed and corpus. Output: data/test/synthetic_retrieval_queries.json


In [36]:
ROOT = PROCESSED.parents[1]
SAMPLE = ROOT / "data" / "samples" / "recipes_sample.csv"
QUERIES_OUT = ROOT / "data" / "test" / "synthetic_retrieval_queries.json"

def canonical_ingredients(ner_field: str, resolver: Resolver) -> list[str]:
    try:
        terms = ast.literal_eval(ner_field)
    except (ValueError, SyntaxError):
        return []
    seen: dict[str, None] = {}
    for term in terms:
        canonical = resolver.resolve(term)
        if canonical:
            seen.setdefault(canonical, None)
    return list(seen)

def run_build_synthetic_queries(corpus=SAMPLE, queries=200, pantry_fraction=0.6, seed=42, min_ingredients=4) -> None:

    corpus = corpus.resolve()
    if not corpus.exists():
        raise SystemExit(f"corpus not found: {corpus}\nRun scripts/download_raw_data.py --recipes sample")
    resolver = Resolver()
    rng = random.Random(seed)

    # Reservoir sample so the full 2.2M-recipe corpus needs O(--queries) memory.
    chosen: list[tuple[int, str, list[str]]] = []
    eligible = 0
    with corpus.open(encoding="utf-8", newline="") as fh:
        for index, row in enumerate(csv.DictReader(fh)):
            ingredients = canonical_ingredients(row["NER"], resolver)
            if len(ingredients) < min_ingredients:
                continue
            candidate = (index, row["title"], ingredients)
            if len(chosen) < queries:
                chosen.append(candidate)
            else:
                slot = rng.randrange(eligible + 1)
                if slot < queries:
                    chosen[slot] = candidate
            eligible += 1

    if not chosen:
        raise SystemExit("no recipes with enough resolvable ingredients")
    chosen.sort(key=lambda c: c[0])
    print(f"[info] {eligible} eligible recipes, {len(chosen)} sampled")

    queries = []
    for index, title, ingredients in chosen:
        count = max(3, round(len(ingredients) * pantry_fraction))
        pantry = rng.sample(ingredients, min(count, len(ingredients) - 1))
        held_out = [i for i in ingredients if i not in pantry]
        queries.append(
            {
                "id": f"syn{len(queries) + 1:04d}",
                "recipe_id": index,
                "title": title,
                "query": "I have " + ", ".join(pantry[:-1]) + (" and " if len(pantry) > 1 else "") + pantry[-1] + ".",
                "pantry": pantry,
                "held_out": held_out,
                "required": ingredients,
            }
        )

    payload = {
        "version": "1.0.0",
        "description": (
            "Synthetic retrieval queries for Pipeline 2 evaluation. recipe_id is the 0-based row "
            "index in the source corpus; required is the full canonical ingredient set of that recipe."
        ),
        "generator": {
            "corpus": str(corpus.relative_to(ROOT)),
            "queries": len(queries),
            "pantry_fraction": pantry_fraction,
            "seed": seed,
            "min_ingredients": min_ingredients,
        },
        "queries": queries,
    }
    QUERIES_OUT.parent.mkdir(parents=True, exist_ok=True)
    QUERIES_OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"[ok] {QUERIES_OUT.relative_to(ROOT)}: {len(queries)} queries from {corpus.name}")


In [37]:
if RUN_BUILD_QUERIES:
    run_build_synthetic_queries()

**`scripts/check_ingredient_coverage.py`**

Validate the curated ingredient tables and report corpus coverage.  Checks (exit code 1 on failure): * every alias canonical exists in the calorie baseline, supplement, or zero list * every pantry-staple ingredient has a calorie treatment * no duplicate aliases after normalization * every annotated-query label resolves to the ingredient vocabulary  Prints how many RecipeNLG `NER` mentions resolve to a calorie source and the most frequent unresolved terms (the next alias/supplement candidates).  python scripts/check_ingredient_coverage.py python scripts/check_ingredient_coverage.py --corpus data/raw/recipenlg/RecipeNLG.csv


In [38]:
COVERAGE_SAMPLE = PROCESSED.parent / "samples" / "recipes_sample.csv"

def run_check_ingredient_coverage(corpus=COVERAGE_SAMPLE, top=25) -> None:

    resolver = Resolver()
    failures: list[str] = []

    alias_rows = read_rows(PROCESSED / "ingredient_aliases.csv")
    for alias, count in Counter(normalize(r["alias"]) for r in alias_rows).items():
        if count > 1:
            failures.append(f"duplicate alias: {alias}")
    for row in alias_rows:
        if normalize(row["canonical"]) not in resolver.known:
            failures.append(f"alias {row['alias']!r} -> unknown canonical {row['canonical']!r}")
    for row in read_rows(PROCESSED / "pantry_staples.csv"):
        if resolver.resolve(row["ingredient"]) is None:
            failures.append(f"pantry staple {row['ingredient']!r} has no calorie treatment")

    annotations = json.loads((PROCESSED.parent / "test" / "annotated_queries.json").read_text(encoding="utf-8"))
    for query in annotations["queries"]:
        for field in ("available_ingredients", "excluded_ingredients"):
            for ingredient in query["expected"][field]:
                if resolver.resolve(ingredient) is None:
                    failures.append(f"{query['id']}: labeled {field} {ingredient!r} is outside the vocabulary")

    unresolved: Counter[str] = Counter()
    total = hit = 0
    if corpus.exists():
        with corpus.open(encoding="utf-8", newline="") as fh:
            for row in csv.DictReader(fh):
                try:
                    terms = ast.literal_eval(row["NER"])
                except (ValueError, SyntaxError):
                    continue
                for term in terms:
                    total += 1
                    if resolver.resolve(term):
                        hit += 1
                    else:
                        unresolved[term.lower().strip()] += 1
    else:
        print(f"[warn] corpus not found: {corpus}", file=sys.stderr)

    print(
        f"baseline={len(resolver.baseline)} supplement={len(resolver.supplement)} "
        f"zero={len(resolver.zero)} aliases={len(resolver.aliases)}"
    )
    if total:
        print(f"corpus coverage: {hit}/{total} ingredient mentions resolved ({hit / total:.1%})")
        print(f"distinct unresolved terms: {len(unresolved)}")
        for term, count in unresolved.most_common(top):
            print(f"  {count:>5}  {term}")

    if failures:
        print("\nFAIL", file=sys.stderr)
        for line in failures:
            print(f"  - {line}", file=sys.stderr)
        raise SystemExit(1)
    print("\nOK: curated tables are internally consistent")


In [39]:
if RUN_CHECK_COVERAGE:
    run_check_ingredient_coverage()             # exit non-zero jika tabel inkonsisten

**`scripts/generate_ner_training_data.py`**

Generate synthetic pantry queries with ingredient spans for NER training.  Recipes from the RecipeNLG sample (``NER`` field) supply real ingredient names; templates turn them into natural-language queries with calorie/exclusion clauses. The matching input is the *masked* context Pipeline 1 actually feeds to its matcher (calorie and exclusion spans blanked by the same production functions), so train-time and inference-time text distributions match.  Output: JSONL rows ``{"text", "spans", "canonical", "query"}`` where ``spans`` are character offsets of ingredient words inside the masked ``text``. Labels are aligned to model subtokens at training time.  Usage: python scripts/generate_ner_training_data.py --count 15000 --seed 42


In [40]:
NER_DATA_OUT_DIR = ROOT / "artifacts" / "ner_training"

CLAUSE_TEMPLATES = [
    "I have {ings}.",
    "I've got {ings}.",
    "I have {ings} in my pantry.",
    "I have {ings} on hand.",
    "i have {ings} at home.",
    "Currently I have {ings}.",
]
CONSTRAINT_TEMPLATES = [
    "",
    "Around {n} calories.",
    "around {n} calories",
    "About {n} kcal.",
    "~{n} kcal",
    "Under {n} calories.",
    "under {n} calories",
    "Max {n} kcal.",
    "less than {n} calories",
    "Between {a} and {b} calories.",
    "At least {a} calories.",
    "No {x}.",
    "no {x}",
    "Without {x}.",
    "No {x} or {y}.",
    "without {x} and {y}",
    "Around {n} calories, no {x}.",
    "Under {n} calories without {x}.",
]

def _join_names(names: list[str], rng: random.Random) -> str:
    if len(names) == 1:
        return names[0]
    if len(names) == 2:
        return f"{names[0]} {rng.choice(['and', 'plus', 'with'])} {names[1]}"
    return ", ".join(names[:-1]) + f" {rng.choice(['and', 'plus'])} {names[-1]}"

def _maybe_pluralize(name: str, rng: random.Random) -> str:
    if rng.random() < 0.30 and not name.endswith("s"):
        head, _, rest = name.rpartition(" ")
        head = head + "s" if not rest else head
        return f"{head} {rest}".strip() if rest else head
    return name

def _maybe_typo(word: str, rng: random.Random) -> str:
    if len(word) >= 4 and rng.random() < 0.10:
        cut = rng.randrange(1, len(word) - 1)
        return word[:cut] + word[cut + 1] + word[cut] + word[cut + 2 :]
    return word

def _surface(name: str, rng: random.Random) -> str:
    """Render a canonical ingredient as a plausible query surface form."""
    words = [_maybe_typo(w, rng) for w in _maybe_pluralize(name, rng).split()]
    return " ".join(words)

def _canonical_pool(vocab: Vocabulary, recipes: list[list[str]]) -> list[str]:
    seen: dict[str, None] = {}
    for names in recipes:
        for name in names:
            canonical = vocab.canonical(normalize_ingredient(name))
            if canonical:
                seen.setdefault(canonical, None)
    return sorted(seen)

def _word_spans(clause: str, names: list[str]) -> list[tuple[int, int]]:
    """Char spans of every expected ingredient word inside the normalized clause."""
    expected = [word for name in names for word in name.split()]
    spans: list[tuple[int, int]] = []
    cursor = 0
    for word in expected:
        match = re.compile(rf"(?<![a-z0-9]){re.escape(word)}(?![a-z0-9])").search(clause, cursor)
        if match is None:  # typo augmentation changed the surface; search all remaining words
            match = re.compile(rf"(?<![a-z0-9]){re.escape(word)}(?![a-z0-9])").search(clause)
            if match is None:
                raise ValueError(f"ingredient word {word!r} not found in clause {clause!r}")
        spans.append((match.start(), match.end()))
        cursor = match.end()
    return spans

def _ingredient_spans(clause: str, names: list[str]) -> list[list[int]]:
    """One char span per ingredient PHRASE (multi-word names stay one entity).

    Per-ingredient grouping is essential: after normalization, consecutive
    different ingredients are also separated by a single space, so a gap-based
    merge would fuse ``chicken potato`` into one entity and corrupt BIO labels.
    """
    word_spans = _word_spans(clause, names)
    phrases: list[list[int]] = []
    cursor = 0
    for name in names:
        width = len(name.split())
        first, last = word_spans[cursor], word_spans[cursor + width - 1]
        phrases.append([first[0], last[1]])
        cursor += width
    return phrases

def generate_one(names: list[str], excluded_pool: list[str], rng: random.Random, vocab: Vocabulary) -> dict:
    surfaces = [_surface(name, rng) for name in names]
    clause = rng.choice(CLAUSE_TEMPLATES).format(ings=_join_names(surfaces, rng))

    excluded = [n for n in rng.sample(excluded_pool, k=min(2, len(excluded_pool))) if n not in names]
    template = rng.choice(CONSTRAINT_TEMPLATES)
    constraint = template.format(
        n=rng.choice(range(200, 800, 25)),
        a=rng.choice(range(250, 500, 50)),
        b=rng.choice(range(500, 800, 50)),
        x=excluded[0] if excluded else "cheese",
        y=excluded[1] if len(excluded) > 1 else "milk",
    )
    query = f"{clause} {constraint}".strip()

    # Production masking, verbatim from query_parser.parse_query.
    text = normalize_text(query)
    _, calorie_spans = extract_calorie_constraints(text)
    masked = _blank(text, calorie_spans)
    spans = exclusion_spans(masked)
    available_context = _blank(masked, spans)

    clause_text = normalize_text(clause)
    canonical = [vocab.canonical(normalize_ingredient(s)) or s for s in surfaces]
    return {
        "text": available_context,
        "spans": _ingredient_spans(clause_text, surfaces),
        "canonical": canonical,
        "query": query,
    }

def run_generate_ner_training_data(count=15000, dev_rows=500, seed=42, corpus=SAMPLE_CSV, output_dir=NER_DATA_OUT_DIR) -> None:

    rng = random.Random(seed)
    vocab = load_vocabulary()

    recipes: list[list[str]] = []
    with corpus.open(encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            names = ast.literal_eval(row["NER"])
            if isinstance(names, list):
                recipes.append([str(n).strip().lower() for n in names if str(n).strip()])

    pool = _canonical_pool(vocab, recipes)
    print(f"canonical ingredient pool: {len(pool)} | recipes: {len(recipes)}")

    rows: list[dict] = []
    attempts = 0
    while len(rows) < count + dev_rows and attempts < (count + dev_rows) * 3:
        attempts += 1
        recipe = rng.choice(recipes)
        usable = [n for n in recipe if vocab.canonical(normalize_ingredient(n))]
        if len(usable) < 2:
            continue
        k = rng.randint(2, min(6, len(usable)))
        names = rng.sample(usable, k=k)
        try:
            rows.append(generate_one(names, pool, rng, vocab))
        except ValueError:
            continue

    rng.shuffle(rows)
    dev, train = rows[: dev_rows], rows[dev_rows :]

    output_dir.mkdir(parents=True, exist_ok=True)
    for split, split_rows in (("train", train), ("dev", dev)):
        out_path = output_dir / f"{split}.jsonl"
        with out_path.open("w", encoding="utf-8") as fh:
            for row in split_rows:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"{out_path}: {len(split_rows)} rows")

    sample = train[0]
    print("sample text:", sample["text"])
    print("sample spans:", sample["spans"], "->", [sample["text"][s:e] for s, e in sample["spans"]])


In [41]:
if RUN_GENERATE_NER_DATA:
    run_generate_ner_training_data()            # -> artifacts/ner_training/{train,dev}.jsonl

**`scripts/train_ner.py`**

Fine-tune DistilBERT for pantry-ingredient token classification (Pipeline 1 NER approach).  Trains on the synthetic masked-context queries produced by ``scripts/generate_ner_training_data.py`` (labels: O / B-ING / I-ING). Best checkpoint by dev span-F1 is saved to ``artifacts/ner_model/`` for the ``NerApproach`` registry entry and the Streamlit UI.  Defaults target Apple-silicon MPS; falls back to CPU automatically.  Usage: python scripts/train_ner.py                       # full run (15k rows, 3 epochs) python scripts/train_ner.py --max-samples 200     # smoke run


In [42]:
DEFAULT_TRAIN = REPO_ROOT / "artifacts" / "ner_training" / "train.jsonl"
DEFAULT_DEV = REPO_ROOT / "artifacts" / "ner_training" / "dev.jsonl"
DEFAULT_OUTPUT = REPO_ROOT / "artifacts" / "ner_model"

LABELS = ["O", "B-ING", "I-ING"]

def _align_labels(text: str, spans: list[list[int]], offsets: list[tuple[int, int]]) -> list[int]:
    """BIO labels per subtoken; -100 for special tokens.

    ``spans`` from the generator are per-ingredient PHRASE spans (multi-word
    names arrive as one span), so no gap-based merging happens here: merging on
    single-space gaps would fuse consecutive distinct ingredients after text
    normalization and corrupt the labels.
    """
    labels: list[int] = []
    for start, end in offsets:
        if end <= start:
            labels.append(-100)
            continue
        entity = next((m for m in spans if start < m[1] and end > m[0]), None)
        if entity is None:
            labels.append(0)
        elif start == entity[0]:
            labels.append(1)  # B-ING
        else:
            labels.append(2)  # I-ING
    return labels

def _load_rows(path: Path, limit: int | None) -> list[dict]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if limit:
        rows = rows[:limit]
    return rows

def _encode(rows: list[dict], tokenizer) -> list[dict[str, list[int]]]:
    encoded = []
    for row in rows:
        item = tokenizer(
            row["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            return_offsets_mapping=True,
        )
        labels = _align_labels(row["text"], row["spans"], item["offset_mapping"])
        encoded.append(
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
                "labels": labels,
                "spans": row["spans"],
            }
        )
    return encoded

def _collate(batch: list[dict[str, list[int]]], tokenizer):
    model_inputs = tokenizer.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    width = model_inputs["input_ids"].shape[1]
    labels = np.full((len(batch), width), -100, dtype=np.int64)
    for i, example in enumerate(batch):
        labels[i, : len(example["labels"])] = example["labels"]
    model_inputs["labels"] = __import__("torch").from_numpy(labels)
    spans = [example["spans"] for example in batch]
    return model_inputs, spans

def _decode_spans(label_ids: list[int], offsets: list[tuple[int, int]]) -> list[list[int]]:
    """BIO decode over real (non-special) subtoken offsets -> char spans."""
    spans: list[list[int]] = []
    open_start: int | None = None
    open_end = 0
    for label, (start, end) in zip(label_ids, offsets):
        if end <= start:
            continue
        if label == 1:  # B-ING
            if open_start is not None:
                spans.append([open_start, open_end])
            open_start, open_end = start, end
        elif label == 2:  # I-ING
            if open_start is None:
                open_start, open_end = start, end
            else:
                open_end = end
        else:
            if open_start is not None:
                spans.append([open_start, open_end])
                open_start = None
    if open_start is not None:
        spans.append([open_start, open_end])
    return spans

def evaluate(model, rows: list[dict[str, list[int]]], tokenizer, device, batch_size: int) -> dict[str, float]:
    import torch

    model.eval()
    token_tp = token_fp = token_fn = 0
    span_tp = span_fp = span_fn = 0
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            batch, _ = _collate(rows[i : i + batch_size], tokenizer)
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = logits.argmax(dim=-1).cpu()

            for j, example in enumerate(rows[i : i + batch_size]):
                offsets = example["offsets"]
                pred_ids = preds[j][: len(offsets)].tolist()
                pred_labels = [p for p, off in zip(pred_ids, offsets) if off[1] > off[0]]
                pred_spans = _decode_spans(
                    pred_labels,
                    [off for off in offsets if off[1] > off[0]],
                )
                gold_labels = [l for l, off in zip(example["labels"], offsets) if off[1] > off[0] and l != -100]
                gold_spans = example["spans"]

                token_tp += sum(1 for p, g in zip(pred_labels, gold_labels) if p != 0 and g != 0 and p == g)
                token_fp += sum(1 for p, g in zip(pred_labels, gold_labels) if p != 0 and g == 0)
                token_fn += sum(1 for p, g in zip(pred_labels, gold_labels) if p == 0 and g != 0)

                pred_set = {tuple(s) for s in pred_spans}
                gold_set = {tuple(s) for s in gold_spans}
                span_tp += len(pred_set & gold_set)
                span_fp += len(pred_set - gold_set)
                span_fn += len(gold_set - pred_set)

    def _f1(tp: int, fp: int, fn: int) -> float:
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "token_f1": _f1(token_tp, token_fp, token_fn),
        "span_f1": _f1(span_tp, span_fp, span_fn),
    }

def run_train_ner(train=DEFAULT_TRAIN, dev=DEFAULT_DEV, output_dir=DEFAULT_OUTPUT, model_name='distilbert-base-uncased', epochs=3, batch_size=32, lr=5e-05, max_samples=None, seed=42) -> None:

    import torch
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = _pick_device(torch)
    print(f"device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    id2label = {0: "O", 1: "B-ING", 2: "I-ING"}
    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=3, id2label=id2label, label2id={v: k for k, v in id2label.items()}
    ).to(device)

    train_rows = _load_rows(train, max_samples)
    dev_rows = _load_rows(dev, None)
    print(f"train rows: {len(train_rows)} | dev rows: {len(dev_rows)}")

    # Keep per-example offsets so evaluation can decode predicted spans.
    train_encoded = _encode(train_rows, tokenizer)
    dev_encoded = _encode(dev_rows, tokenizer)
    for example, row in zip(dev_encoded, dev_rows):
        enc = tokenizer(row["text"], truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True)
        example["offsets"] = enc["offset_mapping"]

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = (len(train_encoded) // batch_size + 1) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_span_f1 = -1.0
    order = list(range(len(train_encoded)))
    for epoch in range(epochs):
        random.Random(seed + epoch).shuffle(order)
        model.train()
        running = 0.0
        for step, start in enumerate(range(0, len(order), batch_size)):
            batch_idx = order[start : start + batch_size]
            batch, _ = _collate([train_encoded[i] for i in batch_idx], tokenizer)
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running += loss.item()
            if step % 50 == 0:
                print(f"epoch {epoch + 1}/{epochs} step {step}: loss={loss.item():.4f}", flush=True)
        metrics = evaluate(model, dev_encoded, tokenizer, device, batch_size)
        print(
            f"epoch {epoch + 1} done | train_loss={running / (len(order) / batch_size + 1):.4f} "
            f"| dev token_f1={metrics['token_f1']:.4f} span_f1={metrics['span_f1']:.4f}",
            flush=True,
        )
        if metrics["span_f1"] > best_span_f1:
            best_span_f1 = metrics["span_f1"]
            output_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            (output_dir / "eval_report.json").write_text(
                json.dumps({"epoch": epoch + 1, **metrics}, indent=2), encoding="utf-8"
            )
            print(f"  saved new best (span_f1={best_span_f1:.4f}) to {output_dir}")

    print(f"done; best dev span_f1={best_span_f1:.4f}")


In [43]:
if RUN_TRAIN_NER:
    run_train_ner()                             # -> artifacts/ner_model/ (best by dev span_f1)

**`scripts/simulate_pipeline2.py`**


In [44]:
DUMMY_QUERIES = [
    {
        "available_ingredients": ["chicken", "potato", "carrot", "onion", "garlic"],
        "excluded_ingredients": ["cheese"],
        "calorie_target": 500,
        "calorie_min": None,
        "calorie_max": None,
    },
    {
        "available_ingredients": ["egg", "milk", "butter", "flour", "sugar"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": 600,
    },
    {
        "available_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
        "excluded_ingredients": ["milk"],
        "calorie_target": None,
        "calorie_min": 400,
        "calorie_max": 700,
    },
]

def run_simulate_pipeline2() -> None:
    retriever = CandidateRetriever.from_artifacts()
    results = []
    for query in DUMMY_QUERIES:
        candidates = retriever.retrieve(query, top_n=5)
        results.append(
            {
                "dummy_structured_query": query,
                "top_candidates": [
                    {
                        "recipe_id": c["recipe_id"],
                        "title": c["title"],
                        "tfidf_score": c["tfidf_score"],
                        "total_calories": c["total_calories"],
                        "canonical_ingredients": c["canonical_ingredients"],
                    }
                    for c in candidates
                ],
            }
        )
    print(json.dumps(results, indent=2))


In [45]:
if RUN_SIMULATE:
    run_simulate_pipeline2()

**`scripts/encode_dense_corpus.py`**

Encode the processed recipe corpus with a sentence-transformer (Pipeline 2 dense approach).  Reads ``data/processed/recipes.parquet`` (same corpus as the TF-IDF approach), encodes each recipe's canonical ingredient text, and stores L2-normalized vectors plus a metadata sidecar under ``artifacts/``.  Usage: python scripts/encode_dense_corpus.py                       # default model + corpus python scripts/encode_dense_corpus.py --model all-MiniLM-L6-v2


In [46]:
DEFAULT_CORPUS = REPO_ROOT / "data" / "processed" / "recipes.parquet"
DEFAULT_OUT_DIR = REPO_ROOT / "artifacts"

def encode_corpus(corpus_path: Path, model_name: str, out_dir: Path, batch_size: int = 128) -> None:
    import pandas as pd
    from sentence_transformers import SentenceTransformer

    corpus = pd.read_parquet(corpus_path)

    def to_text(row) -> str:
        text = str(row.get("canonical_text") or "").strip()
        if not text:
            text = " ".join(row["canonical_ingredients"])
        return text

    texts = [to_text(row) for _, row in corpus.iterrows()]
    print(f"encoding {len(texts)} recipes with {model_name}")

    model = SentenceTransformer(model_name)
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "dense_vectors.npy", vectors)
    meta = {
        "model": model_name,
        "corpus_path": str(corpus_path),
        "column": "canonical_text",
        "normalized": True,
        "dim": int(vectors.shape[1]),
        "n_recipes": int(vectors.shape[0]),
    }
    (out_dir / "dense_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"saved {vectors.shape} vectors to {out_dir / 'dense_vectors.npy'}")
    print(json.dumps(meta, indent=2))

def run_encode_dense_corpus(corpus=DEFAULT_CORPUS, model='all-MiniLM-L6-v2', out_dir=DEFAULT_OUT_DIR, batch_size=128) -> None:
    encode_corpus(corpus, model, out_dir, batch_size=batch_size)


In [47]:
if RUN_ENCODE_DENSE:
    run_encode_dense_corpus()                   # -> artifacts/dense_vectors.npy + dense_meta.json

## 6. Evaluasi & Eksperimen — `evaluation/`

Harness P1 (ekstraksi & perbandingan pendekatan), P2 (retrieval + analisis error),
dan P3 (ranking/ablation). Hasil historis: `artifacts/p2_experiment_results.json`,
`evaluation/results_pipeline3.{json,csv}`.


**`evaluation/evaluate_extraction.py`**

Pipeline 1 evaluation against data/test/annotated_queries.json (PRD §31).  python -m evaluation.evaluate_extraction python -m evaluation.evaluate_extraction --verbose


In [48]:
QUERIES = ROOT / "data" / "test" / "annotated_queries.json"

def _scores(pairs: list[tuple[set[str], set[str]]]) -> tuple[float, float, float, int, int, int]:
    tp = sum(len(pred & gold) for pred, gold in pairs)
    fp = sum(len(pred - gold) for pred, gold in pairs)
    fn = sum(len(gold - pred) for pred, gold in pairs)
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1, tp, fp, fn

def run_evaluate_extraction(queries=QUERIES, verbose=False) -> None:

    cases = json.loads(queries.read_text(encoding="utf-8"))["queries"]
    available: list[tuple[set[str], set[str]]] = []
    excluded: list[tuple[set[str], set[str]]] = []
    calorie_hits = {"calorie_target": 0, "calorie_min": 0, "calorie_max": 0}
    exact = 0
    mismatches = []

    for case in cases:
        expected = case["expected"]
        predicted = parse_query(case["query"])
        available.append((set(predicted["available_ingredients"]), set(expected["available_ingredients"])))
        excluded.append((set(predicted["excluded_ingredients"]), set(expected["excluded_ingredients"])))
        for field in calorie_hits:
            calorie_hits[field] += predicted[field] == expected[field]
        if predicted == expected:
            exact += 1
        else:
            mismatches.append((case["id"], case["query"], expected, predicted))

    for label, pairs in (("available_ingredients", available), ("excluded_ingredients", excluded)):
        precision, recall, f1, tp, fp, fn = _scores(pairs)
        print(f"{label:<24} P={precision:.3f} R={recall:.3f} F1={f1:.3f}  (tp={tp} fp={fp} fn={fn})")

    total = len(cases)
    for field, hits in calorie_hits.items():
        print(f"{field:<24} accuracy={hits / total:.3f} ({hits}/{total})")
    print(f"{'structured query exact':<24} {exact}/{total} ({exact / total:.1%})")

    if verbose:
        for query_id, query, expected, predicted in mismatches:
            print(f"\n{query_id} | {query}")
            for field in expected:
                if expected[field] != predicted[field]:
                    print(f"  {field}: expected {expected[field]} got {predicted[field]}")


**`evaluation/evaluate_retrieval.py`**


In [49]:
DEFAULT_QUERIES_PATH = ROOT / "data" / "test" / "synthetic_retrieval_queries.json"

def generate_in_memory_queries(
    corpus: pd.DataFrame,
    num_queries: int = 200,
    pantry_fraction: float = 0.6,
    min_ingredients: int = 4,
    seed: int = 42,
) -> list[dict[str, object]]:
    rng = random.Random(seed)
    eligible = [
        (int(row["recipe_id"]), str(row["title"]), list(row["canonical_ingredients"]))
        for _, row in corpus.iterrows()
        if len(row["canonical_ingredients"]) >= min_ingredients
    ]
    if not eligible:
        return []
    sampled = rng.sample(eligible, min(num_queries, len(eligible)))
    queries: list[dict[str, object]] = []
    for idx, (recipe_id, title, ingredients) in enumerate(sampled, start=1):
        count = max(3, round(len(ingredients) * pantry_fraction))
        pantry = rng.sample(ingredients, min(count, len(ingredients) - 1))
        held_out = [item for item in ingredients if item not in pantry]
        queries.append(
            {
                "id": f"sim{idx:04d}",
                "recipe_id": recipe_id,
                "title": title,
                "pantry": pantry,
                "held_out": held_out,
                "required": ingredients,
            }
        )
    return queries

def rank_target(retriever: CandidateRetriever, item: dict[str, object], max_k: int) -> int | None:
    """1-based rank of the target recipe in the retriever's top-``max_k``, or ``None``.

    Shared by ``evaluate_retriever`` and the Pipeline 2 experiment harness so the
    retrieval loop is written once.
    """
    structured_query = {
        "available_ingredients": list(item["pantry"]),
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
    indices, _ = retriever.retrieve_indices(structured_query, top_n=max_k)
    for pos, idx in enumerate(indices, start=1):
        if int(retriever.recipe_ids[idx]) == int(item["recipe_id"]):
            return pos
    return None

def evaluate_retriever(
    retriever: CandidateRetriever,
    queries: list[dict[str, object]],
    ks: tuple[int, ...] = (1, 5, 10, 20, 50),
) -> dict[str, float]:
    max_k = max(ks)
    hits_at_k = {k: 0 for k in ks}
    precision_at_k = {k: 0.0 for k in ks}
    reciprocal_ranks: list[float] = []

    valid_ids = set(int(rid) for rid in retriever.recipe_ids)
    evaluated = 0

    for item in queries:
        if int(item["recipe_id"]) not in valid_ids:
            continue
        evaluated += 1
        rank = rank_target(retriever, item, max_k)
        if rank is not None:
            reciprocal_ranks.append(1.0 / rank)
            for k in ks:
                if rank <= k:
                    hits_at_k[k] += 1
                    precision_at_k[k] += 1.0 / k
        else:
            reciprocal_ranks.append(0.0)

    if evaluated == 0:
        return {"evaluated_queries": 0.0, "mrr": 0.0}

    metrics: dict[str, float] = {"evaluated_queries": float(evaluated)}
    for k in ks:
        metrics[f"recall@{k}"] = round(hits_at_k[k] / evaluated, 4)
        metrics[f"precision@{k}"] = round(precision_at_k[k] / evaluated, 4)
    metrics["mrr"] = round(sum(reciprocal_ranks) / evaluated, 4)
    return metrics

def run_evaluate_retrieval(corpus=DEFAULT_CORPUS_PARQUET, vectorizer=DEFAULT_VECTORIZER_PATH, vectors=DEFAULT_VECTORS_PATH, queries_path=DEFAULT_QUERIES_PATH, sample_queries=200) -> None:

    retriever = CandidateRetriever.from_artifacts(
        corpus_path=corpus,
        vectorizer_path=vectorizer,
        vectors_path=vectors,
    )

    queries: list[dict[str, object]] = []
    if queries_path.exists():
        payload = json.loads(queries_path.read_text(encoding="utf-8"))
        queries = payload.get("queries", [])

    metrics = evaluate_retriever(retriever, queries)
    if metrics.get("evaluated_queries", 0.0) == 0.0:
        queries = generate_in_memory_queries(retriever.corpus, num_queries=sample_queries)
        metrics = evaluate_retriever(retriever, queries)

    print(json.dumps(metrics, indent=2))


**`evaluation/experiment.py`**

Compare Pipeline 1 ingredient-matching approaches and report their errors.  Run ``python -m evaluation.experiment``; add ``--verbose`` for query-level diffs and ``--output-json PATH`` to persist the metrics table (used by the Streamlit comparison tab). All variants share the production calorie/exclusion parser so this experiment isolates ingredient matching. Approaches come from the ``src.pipeline1.approaches`` registry (single source of truth); no LLM is used.


In [50]:
DATASETS = {
    "annotated": ROOT / "data/test/annotated_queries.json",
    "stress": ROOT / "data/test/stress_queries.json",
}
CALORIE_FIELDS = ("calorie_target", "calorie_min", "calorie_max")
FIELDS = ("available_ingredients", "excluded_ingredients", *CALORIE_FIELDS)

def _errors(expected: dict, predicted: dict) -> list[tuple[str, str]]:
    errors = []
    for field, category in (("available_ingredients", "ingredient"), ("excluded_ingredients", "exclusion")):
        gold, got = set(expected[field]), set(predicted[field])
        errors.extend((f"{category}_miss", item) for item in sorted(gold - got))
        errors.extend((f"{category}_extra", item) for item in sorted(got - gold))

    swapped_fields = set()
    for field in CALORIE_FIELDS:
        value = expected[field]
        if value is not None and predicted[field] != value:
            other = next((name for name in CALORIE_FIELDS if name != field and predicted[name] == value), None)
            if other:
                errors.append(("calorie_field", f"{field}->{other}"))
                swapped_fields.add(other)
            elif predicted[field] is None:
                errors.append(("calorie_miss", field))
            else:
                errors.append(("calorie_value", field))
    for field in CALORIE_FIELDS:
        if field not in swapped_fields and expected[field] is None and predicted[field] is not None:
            errors.append(("calorie_extra", field))
    return errors

def _f1(pairs: list[tuple[set[str], set[str]]]) -> float:
    tp = sum(len(pred & gold) for pred, gold in pairs)
    fp = sum(len(pred - gold) for pred, gold in pairs)
    fn = sum(len(gold - pred) for pred, gold in pairs)
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def _approach_names() -> list[str]:
    """All registry approaches with loadable artifacts, in registry order."""
    return [entry["name"] for entry in available_approaches() if entry["available"]]

def run_experiment(
    approach_names: list[str] | None = None,
    datasets: dict[str, Path] | None = None,
    vocab: Vocabulary | None = None,
) -> dict[str, dict]:
    """Evaluate each approach over the annotated + stress fixtures.

    Returns ``{approach: {total, exact, exact_pct, available_f1, excluded_f1,
    calorie_accuracy, errors, failures}}`` — the single source both the CLI
    table and the Streamlit comparison tab render.
    """
    datasets = datasets or DATASETS
    vocab = vocab or load_vocabulary()
    names = approach_names or _approach_names()
    cases = [(name, item) for name, path in datasets.items() for item in json.loads(path.read_text(encoding="utf-8"))["queries"]]

    results: dict[str, dict] = {}
    for name in names:
        approach = get_approach(name)
        exact = 0
        available: list[tuple[set[str], set[str]]] = []
        excluded: list[tuple[set[str], set[str]]] = []
        calorie_hits = 0
        taxonomy: Counter[str] = Counter()
        failures = []
        for dataset, case in cases:
            expected = case["expected"]
            predicted = parse_query(case["query"], vocab, approach.extract)
            errs = _errors(expected, predicted)
            taxonomy.update(category for category, _ in errs)
            exact += predicted == expected
            available.append((set(predicted["available_ingredients"]), set(expected["available_ingredients"])))
            excluded.append((set(predicted["excluded_ingredients"]), set(expected["excluded_ingredients"])))
            calorie_hits += sum(predicted[field] == expected[field] for field in CALORIE_FIELDS)
            if errs:
                failures.append(
                    {
                        "dataset": dataset,
                        "id": case["id"],
                        "query": case["query"],
                        "expected": expected,
                        "predicted": predicted,
                        "errors": errs,
                    }
                )
        total = len(cases)
        results[name] = {
            "total": total,
            "exact": exact,
            "exact_pct": exact / total if total else 0.0,
            "available_f1": round(_f1(available), 4),
            "excluded_f1": round(_f1(excluded), 4),
            "calorie_accuracy": round(calorie_hits / (total * 3), 4) if total else 0.0,
            "errors": dict(taxonomy),
            "failures": failures,
        }
    return results

def format_results_table(results: dict[str, dict]) -> str:
    """ASCII-aligned table, same layout the report historically used."""
    lines = [
        "Approach         Exact     Available F1  Excluded F1  Calorie accuracy",
        "---------------- --------- ------------- ------------ ----------------",
    ]
    for name, metrics in results.items():
        lines.append(
            f"{name:<16} {metrics['exact']:>3}/{metrics['total']:<5}  {metrics['available_f1']:>11.3f}  "
            f"{metrics['excluded_f1']:>10.3f}  {metrics['calorie_accuracy']:>15.3f}"
        )
        errors = ", ".join(f"{key}={value}" for key, value in sorted(metrics["errors"].items())) or "none"
        lines.append(f"  errors: {errors}")
    return "\n".join(lines)

def run_p1_experiment(verbose=False, output_json=None) -> None:

    results = run_experiment()
    print(f"Pipeline 1: {results[next(iter(results))]['total']} queries (annotated + stress)")
    print(format_results_table(results))

    if verbose:
        for name, metrics in results.items():
            for failure in metrics["failures"]:
                print(f"  {name} {failure['dataset']}/{failure['id']}: {failure['query']}")
                print(f"    errors={failure['errors']}; expected={failure['expected']}; predicted={failure['predicted']}")

    if output_json:
        output_json.parent.mkdir(parents=True, exist_ok=True)
        output_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"[info] wrote {output_json}")


**`evaluation/experiment_pipeline2.py`**


In [51]:
DEFAULT_OUT_JSON = ROOT / "artifacts" / "p2_experiment_results.json"
DEFAULT_OUT_CHART = ROOT / "artifacts" / "p2_error_analysis.png"

def _load_staples(path: Path = DEFAULT_STAPLES_PATH) -> set[str]:
    if not path.exists():
        return set()
    with path.open(encoding="utf-8", newline="") as fh:
        return {normalize_ingredient(r["ingredient"]) for r in csv.DictReader(fh)}

def _build_approaches(corpus: pd.DataFrame) -> list[tuple[str, str, object, set[str] | None]]:
    documents = corpus["canonical_text"].fillna("").astype(str).tolist()
    specs = [
        (
            "tfidf-raw",
            "TF-IDF baseline (max_features=1000, ngram 1-2) tanpa down-weighting pantry staple",
            dict(max_features=1000, ngram_range=(1, 2), sublinear_tf=False, use_staple_downweight=False),
        ),
        (
            "tfidf-staple",
            "TF-IDF produksi (max_features=1000, ngram 1-2) dengan pantry staple down-weighting",
            dict(max_features=1000, ngram_range=(1, 2), sublinear_tf=False, use_staple_downweight=True),
        ),
        (
            "tfidf-unigram",
            "TF-IDF unigram saja (max_features=1000, ngram 1-1) dengan staple down-weighting",
            dict(max_features=1000, ngram_range=(1, 1), sublinear_tf=False, use_staple_downweight=True),
        ),
        (
            "tfidf-unigram-full",
            "TF-IDF unigram kosakata penuh (max_features=None, ngram 1-1, sublinear_tf) + staple down-weight",
            dict(max_features=None, ngram_range=(1, 1), sublinear_tf=True, use_staple_downweight=True),
        ),
    ]

    built: list[tuple[str, str, object, set[str] | None]] = []
    for name, desc, kwargs in specs:
        vec = RecipeVectorizer(**kwargs)
        mat = vec.fit_transform(documents)
        retriever = CandidateRetriever(vectorizer=vec, recipe_vectors=mat, corpus=corpus)
        vocab_terms = set(vec.vectorizer.get_feature_names_out())
        built.append((name, desc, retriever, vocab_terms))

    if DEFAULT_DENSE_VECTORS_PATH.exists() and DEFAULT_DENSE_META_PATH.exists():
        try:
            dense = DenseApproach()
            _ = dense.model
            built.append(
                (
                    "dense-minilm",
                    "Embedding semantik lokal all-MiniLM-L6-v2 (384-dim, L2-normalized) + cosine similarity",
                    dense,
                    None,
                )
            )
        except Exception:
            pass

    return built

def _classify_miss(
    item: dict[str, object],
    vocab_terms: set[str] | None,
    staples: set[str],
) -> str:
    pantry = [str(x) for x in item["pantry"]]
    required = [str(x) for x in item["required"]]

    if vocab_terms is not None:
        query_tokens = query_to_text(pantry).split()
        if any(tok not in vocab_terms for tok in query_tokens):
            return "miss_oov"

    staple_hits = sum(1 for ing in pantry if normalize_ingredient(ing) in staples)
    if pantry and (staple_hits / len(pantry)) >= 0.5:
        return "miss_staple"

    if len(required) >= 8:
        return "miss_long_recipe"

    return "miss_overlap"

def run_p2_experiment(
    corpus_path: Path = DEFAULT_CORPUS_PARQUET,
    num_queries: int = 100,
    pantry_fraction: float = 0.6,
    seed: int = 42,
    miss_cutoff: int = 10,
    ks: tuple[int, ...] = (1, 5, 10, 20, 50),
) -> list[dict[str, object]]:
    corpus = pd.read_parquet(corpus_path).reset_index(drop=True)
    queries = generate_in_memory_queries(
        corpus,
        num_queries=num_queries,
        pantry_fraction=pantry_fraction,
        seed=seed,
    )
    staples = _load_staples()
    approaches = _build_approaches(corpus)
    max_k = max(ks)
    results: list[dict[str, object]] = []

    for name, desc, retriever, vocab_terms in approaches:
        hits = {k: 0 for k in ks}
        rr_list: list[float] = []
        errors = {
            "miss_oov": 0,
            "miss_staple": 0,
            "miss_long_recipe": 0,
            "miss_overlap": 0,
        }
        t0 = time.perf_counter()
        for item in queries:
            rank = rank_target(retriever, item, max_k)

            if rank is not None:
                rr_list.append(1.0 / rank)
                for k in ks:
                    if rank <= k:
                        hits[k] += 1
            else:
                rr_list.append(0.0)

            if rank is None or rank > miss_cutoff:
                cause = _classify_miss(item, vocab_terms, staples)
                errors[cause] += 1

        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        n = max(len(queries), 1)
        row: dict[str, object] = {
            "approach": name,
            "description": desc,
            "queries": len(queries),
            "recall@1": round(hits[1] / n, 4),
            "recall@5": round(hits[5] / n, 4),
            "recall@10": round(hits[10] / n, 4),
            "recall@20": round(hits[20] / n, 4),
            "recall@50": round(hits[50] / n, 4),
            "mrr": round(sum(rr_list) / n, 4),
            "avg_ms": round(elapsed_ms / n, 2),
            "total_misses": sum(errors.values()),
            "errors": errors,
        }
        results.append(row)

    return results

MISS_CATEGORIES = [
    ("miss_oov", "Vocab terpotong (OOV)", "#175935"),
    ("miss_staple", "Dominasi pantry staple", "#2D8659"),
    ("miss_long_recipe", "Resep panjang (>=8 bahan)", "#D6702B"),
    ("miss_overlap", "Kompetitor / overlap leksikal", "#547EB2"),
]

def save_error_chart(
    results: list[dict[str, object]],
    out_path: Path = DEFAULT_OUT_CHART,
    miss_cutoff: int = 10,
) -> Path:
    """Stacked bar chart of retrieval miss causes per approach (matplotlib)."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    sorted_rows = sorted(results, key=lambda r: int(r["total_misses"]), reverse=True)
    labels = [str(r["approach"]) for r in sorted_rows]
    fig, ax = plt.subplots(figsize=(10, max(4.0, 0.62 * len(labels) + 2.5)))

    left = [0] * len(sorted_rows)
    for key, label, color in MISS_CATEGORIES:
        values = [int(r["errors"].get(key, 0)) for r in sorted_rows]
        ax.barh(labels, values, left=left, color=color, label=label)
        left = [l + v for l, v in zip(left, values)]

    ax.invert_yaxis()
    ax.set_title("Dari mana miss retrieval berasal? — Pipeline 2")
    ax.set_xlabel(f"jumlah miss (target gagal masuk Top-{miss_cutoff})")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

def run_p2_experiment_cli(corpus=DEFAULT_CORPUS_PARQUET, queries=100, miss_cutoff=10, out_json=DEFAULT_OUT_JSON, out_chart=DEFAULT_OUT_CHART) -> None:

    results = run_p2_experiment(
        corpus_path=corpus,
        num_queries=queries,
        miss_cutoff=miss_cutoff,
    )
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
    save_error_chart(results, out_path=out_chart, miss_cutoff=miss_cutoff)
    print(json.dumps(results, indent=2))


**`evaluation/evaluate_ranking.py`**

Standalone evaluation & ablation study for Pipeline 3 (PRD Sections 33-35).  Runs independently without waiting for Pipeline 1 & Pipeline 2 by: 1. Reading ground-truth structured queries from ``data/test/annotated_queries.json`` (decoupling Pipeline 3 evaluation from Pipeline 1 parsing errors). 2. Running a self-contained local TF-IDF candidate retriever on ``data/samples/recipes_sample.csv`` (matching PRD §17: TfidfVectorizer with ``max_features=1000, ngram_range=(1, 2)`` + cosine similarity, Top-N=50). 3. Estimating recipe calories on-demand via ``RecipeCalorieEstimator`` using the committed tables in ``data/processed/``. 4. Comparing Baseline vs Proposed and running the full PRD §35 Ablation Study: - Model A: TF-IDF only (Baseline) - Model B: TF-IDF + Ingredient Coverage - Model C: TF-IDF + Ingredient Coverage + Missing Ingredient Penalty - Model D (Full Proposed): TF-IDF + Coverage + Missing Penalty + Calorie Score + Exclusion Filter - Model D (Unweighted): Full Proposed without pantry_staples.csv weights  Usage: python evaluation/evaluate_ranking.py python evaluation/evaluate_ranking.py --top-k 3 --candidate-n 50 --examples 3 python evaluation/evaluate_ranking.py --output-json evaluation/results_pipeline3.json


In [52]:
EVAL_SAMPLE_CORPUS = REPO_ROOT / "data" / "samples" / "recipes_sample.csv"
DEFAULT_QUERIES = REPO_ROOT / "data" / "test" / "annotated_queries.json"
DEFAULT_PROCESSED = REPO_ROOT / "data" / "processed"

@dataclass
class RawRecipeRow:
    recipe_id: int
    title: str
    raw_ingredients: list[str]
    ner_ingredients: list[str]
    canonical_ingredients: list[str]
    directions: list[str]
    ingredient_doc: str

class LocalTfidfRetriever:
    """Self-contained PRD §17 retriever for standalone Pipeline 3 evaluation."""

    def __init__(
        self,
        corpus_path: Path,
        normalizer: IngredientNormalizer,
        max_features: int = 1000,
    ) -> None:
        self.normalizer = normalizer
        self.recipes: list[RawRecipeRow] = []

        with corpus_path.open(encoding="utf-8", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh)):
                try:
                    ner = ast.literal_eval(row["NER"])
                    raw_ings = ast.literal_eval(row["ingredients"])
                    directions = ast.literal_eval(row["directions"])
                except (ValueError, SyntaxError):
                    continue
                canonical = normalizer.canonicalize_list(ner)
                if not canonical:
                    continue
                # Join canonical ingredients into space-separated document (PRD §17.2)
                doc = " ".join(canonical)
                self.recipes.append(
                    RawRecipeRow(
                        recipe_id=idx,
                        title=row["title"].strip(),
                        raw_ingredients=raw_ings,
                        ner_ingredients=ner,
                        canonical_ingredients=canonical,
                        directions=directions,
                        ingredient_doc=doc,
                    )
                )

        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([r.ingredient_doc for r in self.recipes])

    def retrieve(
        self,
        query: StructuredQuery,
        estimator: RecipeCalorieEstimator,
        calorie_cache: dict[int, float | None],
        top_n: int = 50,
        calorie_mode: str = "per_serving",
    ) -> list[CandidateRecipe]:
        query_terms = self.normalizer.canonicalize_list(query.available_ingredients)
        query_doc = " ".join(query_terms)
        if not query_doc:
            return []

        q_vec = self.vectorizer.transform([query_doc])
        sims = cosine_similarity(q_vec, self.matrix).ravel()
        k = min(top_n, len(self.recipes))
        if k <= 0:
            return []

        top_indices = np.argpartition(sims, -k)[-k:]
        top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]

        candidates: list[CandidateRecipe] = []
        for pos in top_indices:
            sim_score = float(sims[pos])
            if sim_score <= 0.0:
                continue
            row = self.recipes[int(pos)]
            if row.recipe_id not in calorie_cache:
                est = estimator.estimate_recipe(
                    row.raw_ingredients,
                    row.ner_ingredients,
                    row.directions,
                    title=row.title,
                )
                calorie_cache[row.recipe_id] = (
                    est.calories_per_serving if calorie_mode == "per_serving" else est.total_calories
                )
            candidates.append(
                CandidateRecipe(
                    recipe_id=row.recipe_id,
                    title=row.title,
                    canonical_ingredients=row.canonical_ingredients,
                    tfidf_score=sim_score,
                    ingredients=row.raw_ingredients,
                    directions=row.directions,
                    total_calories=calorie_cache[row.recipe_id],
                )
            )
        return candidates

def evaluate_model(
    model_name: str,
    config: RankingConfig,
    reranker: MultiCriteriaReranker,
    query_candidates: list[tuple[str, StructuredQuery, list[CandidateRecipe]]],
    top_k: int,
    mrr_coverage_threshold: float = 0.50,
) -> dict[str, Any]:
    """Compute PRD §§33-34 metrics across all evaluated queries for a single config."""
    weighted_covs: list[float] = []
    unweighted_covs: list[float] = []
    missing_totals: list[int] = []
    missing_non_staples: list[int] = []
    calorie_distances: list[float] = []
    constraint_violations = 0
    exclusion_violations = 0
    calorie_bound_violations = 0
    total_recommended = 0
    total_with_exclusion_query = 0
    total_with_calorie_bound_query = 0
    total_with_any_constraint_query = 0
    reciprocal_ranks: list[float] = []
    fallback_queries = 0

    for _qid, query, candidates in query_candidates:
        result = reranker.rerank(candidates, query, top_k=top_k, config=config)
        if result.is_fallback:
            fallback_queries += 1

        recs = result.recommendations
        first_relevant_rr = 0.0

        for rec in recs:
            total_recommended += 1
            weighted_covs.append(rec.coverage.weighted_coverage)
            unweighted_covs.append(rec.coverage.unweighted_coverage)
            missing_totals.append(rec.coverage.missing_count)
            missing_non_staples.append(rec.coverage.missing_non_staple_count)

            if query.has_calorie_constraint and rec.calorie.calorie_distance is not None:
                calorie_distances.append(rec.calorie.calorie_distance)

            excl_bad = bool(query.excluded_ingredients) and rec.exclusion.violated
            cal_bad = query.has_hard_calorie_bound and (not rec.calorie.passed_hard_filter)
            any_bad = excl_bad or cal_bad

            if query.excluded_ingredients:
                total_with_exclusion_query += 1
                if excl_bad:
                    exclusion_violations += 1

            if query.has_hard_calorie_bound:
                total_with_calorie_bound_query += 1
                if cal_bad:
                    calorie_bound_violations += 1

            if query.excluded_ingredients or query.has_hard_calorie_bound:
                total_with_any_constraint_query += 1
                if any_bad:
                    constraint_violations += 1

            # Practical cookability relevance for MRR:
            # no exclusion violation, passes hard calorie bound, and weighted coverage >= threshold
            if first_relevant_rr == 0.0 and (not any_bad) and rec.coverage.weighted_coverage >= mrr_coverage_threshold:
                first_relevant_rr = 1.0 / rec.rank

        reciprocal_ranks.append(first_relevant_rr)

    def _mean(values: list[float] | list[int]) -> float:
        return float(sum(values) / len(values)) if values else 0.0

    return {
        "model": model_name,
        "top_k": top_k,
        "num_queries": len(query_candidates),
        "weighted_coverage_at_k": round(_mean(weighted_covs), 4),
        "unweighted_coverage_at_k": round(_mean(unweighted_covs), 4),
        "avg_missing_at_k": round(_mean(missing_totals), 3),
        "avg_missing_non_staple_at_k": round(_mean(missing_non_staples), 3),
        "calorie_distance": round(_mean(calorie_distances), 1),
        "constraint_violation_rate": round(
            constraint_violations / total_with_any_constraint_query, 4
        ) if total_with_any_constraint_query else 0.0,
        "exclusion_violation_rate": round(
            exclusion_violations / total_with_exclusion_query, 4
        ) if total_with_exclusion_query else 0.0,
        "calorie_bound_violation_rate": round(
            calorie_bound_violations / total_with_calorie_bound_query, 4
        ) if total_with_calorie_bound_query else 0.0,
        "mrr": round(_mean(reciprocal_ranks), 4),
        "fallback_rate": round(fallback_queries / len(query_candidates), 4) if query_candidates else 0.0,
        "total_recommended": total_recommended,
    }

def format_markdown_table(rows: list[dict[str, Any]], top_k: int) -> str:
    headers = [
        "Model",
        f"Weighted Cov@{top_k} ↑",
        f"Unweighted Cov@{top_k} ↑",
        f"Missing@{top_k} ↓",
        f"Non-Staple Miss@{top_k} ↓",
        "Calorie Dist (kcal) ↓",
        "Constraint Viol ↓",
        "Excl Viol ↓",
        "Cal Viol ↓",
        "MRR ↑",
        "Fallback",
    ]
    sep = ["---"] + ["---:"] * (len(headers) - 1)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(sep) + " |"]
    for r in rows:
        lines.append(
            "| "
            + " | ".join([
                r["model"],
                f"{r['weighted_coverage_at_k']:.1%}",
                f"{r['unweighted_coverage_at_k']:.1%}",
                f"{r['avg_missing_at_k']:.2f}",
                f"{r['avg_missing_non_staple_at_k']:.2f}",
                f"{r['calorie_distance']:.1f}",
                f"{r['constraint_violation_rate']:.1%}",
                f"{r['exclusion_violation_rate']:.1%}",
                f"{r['calorie_bound_violation_rate']:.1%}",
                f"{r['mrr']:.3f}",
                f"{r['fallback_rate']:.1%}",
            ])
            + " |"
        )
    return "\n".join(lines)

def run_evaluate_ranking(corpus=EVAL_SAMPLE_CORPUS, queries=DEFAULT_QUERIES, processed_dir=DEFAULT_PROCESSED, top_k=3, candidate_n=50, calorie_mode='per_serving', examples=2, output_json=None, output_csv=None) -> None:

    t0 = time.perf_counter()
    normalizer = IngredientNormalizer(processed_dir)
    staple_weights = PantryStapleWeights.from_csv(
        processed_dir / "pantry_staples.csv",
        normalizer=normalizer,
    )
    estimator = RecipeCalorieEstimator(processed_dir=processed_dir, normalizer=normalizer)
    reranker = MultiCriteriaReranker(
        processed_dir=processed_dir,
        normalizer=normalizer,
        staple_weights=staple_weights,
    )

    print(f"[info] Loading corpus: {corpus}")
    retriever = LocalTfidfRetriever(corpus, normalizer=normalizer)
    print(f"[info] Indexed {len(retriever.recipes):,} recipes with TF-IDF ({time.perf_counter() - t0:.2f}s)")

    raw_queries = json.loads(queries.read_text(encoding="utf-8"))["queries"]
    calorie_cache: dict[int, float | None] = {}
    query_candidates: list[tuple[str, StructuredQuery, list[CandidateRecipe]]] = []

    t_ret = time.perf_counter()
    for entry in raw_queries:
        sq = StructuredQuery.from_dict(entry["expected"], raw_query=entry["query"])
        cands = retriever.retrieve(
            sq,
            estimator=estimator,
            calorie_cache=calorie_cache,
            top_n=candidate_n,
            calorie_mode=calorie_mode,
        )
        query_candidates.append((entry["id"], sq, cands))

    estimated_non_null = sum(1 for v in calorie_cache.values() if v is not None)
    print(
        f"[info] Retrieved Top-{candidate_n} candidates for {len(query_candidates)} queries; "
        f"estimated calories for {estimated_non_null}/{len(calorie_cache)} unique candidate recipes "
        f"({estimated_non_null / max(len(calorie_cache), 1):.1%}) in {time.perf_counter() - t_ret:.2f}s"
    )

    # PRD Section 35 Ablation Study + Pantry Staple Weight validation
    ablation_models: list[tuple[str, RankingConfig]] = [
        ("Model A: TF-IDF Only (Baseline)", RankingConfig.ablation_model_a()),
        ("Model B: TF-IDF + Coverage", RankingConfig.ablation_model_b()),
        ("Model C: TF-IDF + Cov + Missing Pen.", RankingConfig.ablation_model_c()),
        ("Model D: Full Proposed (Weighted)", RankingConfig.ablation_model_d()),
        ("Model D (Unweighted Staples)", RankingConfig.ablation_unweighted()),
    ]

    results = [
        evaluate_model(name, cfg, reranker, query_candidates, top_k=top_k)
        for name, cfg in ablation_models
    ]

    print("\n=== Pipeline 3 Evaluation & Ablation Study (PRD §§33-35) ===")
    print(
        f"Corpus: {corpus.name} | Queries: {queries.name} ({len(query_candidates)}) | "
        f"Top-N={candidate_n} -> Top-K={top_k} | Calorie mode: {calorie_mode}"
    )
    print("Note: MRR measures first Top-K recipe with 0 constraint violations & weighted coverage >= 50%.\n")
    print(format_markdown_table(results, top_k=top_k))

    # Print qualitative Baseline vs Proposed examples for combined-constraint queries
    if examples > 0:
        demo_ids = ("q041", "q042", "q044", "q045", "q056", "q060")
        selected_demos = [qc for qc in query_candidates if qc[0] in demo_ids][: examples]
        if not selected_demos:
            selected_demos = query_candidates[: examples]

        print("\n=== Qualitative Comparison: Baseline (Model A) vs Proposed (Model D) ===")
        for qid, sq, cands in selected_demos:
            base_res = reranker.rerank(cands, sq, top_k=1, config=RankingConfig.ablation_model_a())
            prop_res = reranker.rerank(cands, sq, top_k=1, config=RankingConfig.ablation_model_d())
            print(f"\n[{qid}] Query: {sq.raw_query}")
            print("  -- Baseline #1 (TF-IDF Only) --")
            if base_res.recommendations:
                print(
                    format_recommendation_card(
                        base_res.recommendations[0],
                        excluded_ingredients=sq.excluded_ingredients,
                        include_ingredients=False,
                        include_directions=False,
                    )
                )
            print("\n  -- Proposed #1 (Full Multi-Criteria Reranker) --")
            if prop_res.is_fallback and prop_res.fallback_message:
                print(f"  [NOTICE] {prop_res.fallback_message}")
            if prop_res.recommendations:
                print(
                    format_recommendation_card(
                        prop_res.recommendations[0],
                        excluded_ingredients=sq.excluded_ingredients,
                        include_ingredients=False,
                        include_directions=False,
                    )
                )

    if output_json is not None:
        output_json.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "corpus": str(corpus),
            "queries": str(queries),
            "top_k": top_k,
            "candidate_n": candidate_n,
            "calorie_mode": calorie_mode,
            "unique_candidates_evaluated": len(calorie_cache),
            "calorie_estimation_coverage": round(estimated_non_null / max(len(calorie_cache), 1), 4),
            "ablation_results": results,
        }
        output_json.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        print(f"\n[ok] Saved JSON report -> {output_json}")

    if output_csv is not None:
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        with output_csv.open("w", encoding="utf-8", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(results[0].keys()))
            writer.writeheader()
            writer.writerows(results)
        print(f"[ok] Saved CSV table -> {output_csv}")


In [53]:
if RUN_EVALUATIONS:
    run_evaluate_extraction()                   # P/R/F1 ekstraksi pada 60 query teranotasi
    run_p1_experiment()                         # perbandingan rule/exact/difflib/n-gram
    run_evaluate_retrieval()                    # recall/MRR retrieval TF-IDF
    run_p2_experiment_cli()                     # eksperimen P2 + grafik analisis error
    run_evaluate_ranking()                      # evaluasi ranking Pipeline 3

available_ingredients    P=1.000 R=1.000 F1=1.000  (tp=183 fp=0 fn=0)
excluded_ingredients     P=1.000 R=1.000 F1=1.000  (tp=32 fp=0 fn=0)
calorie_target           accuracy=1.000 (60/60)
calorie_min              accuracy=1.000 (60/60)
calorie_max              accuracy=1.000 (60/60)
structured query exact   60/60 (100.0%)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5813.00it/s]

Pipeline 1: 80 queries (annotated + stress)
Approach         Exact     Available F1  Excluded F1  Calorie accuracy
---------------- --------- ------------- ------------ ----------------
rule              76/80           0.979       0.989            1.000
  errors: exclusion_miss=1, ingredient_miss=10
surface-exact     32/80           0.860       0.831            1.000
  errors: exclusion_miss=13, ingredient_extra=1, ingredient_miss=58
difflib-0.82      78/80           0.996       1.000            1.000
  errors: ingredient_miss=2
char-ngram-0.60   76/80           0.985       0.989            1.000
  errors: exclusion_miss=1, ingredient_miss=7
ner               74/80           0.972       0.989            1.000
  errors: exclusion_miss=1, ingredient_miss=13


{
  "evaluated_queries": 200.0,
  "recall@1": 0.31,
  "precision@1": 0.31,
  "recall@5": 0.53,
  "precision@5": 0.106,
  "recall@10": 0.605,
  "precision@10": 0.0605,
  "recall@20": 0.69,
  "precision@20": 0.0345,
  "recall@50": 0.815,
  "precision@50": 0.0163,
  "mrr": 0.4116
}


... (2 outputs left out)

[
  {
    "approach": "tfidf-raw",
    "description": "TF-IDF baseline (max_features=1000, ngram 1-2) tanpa down-weighting pantry staple",
    "queries": 100,
    "recall@1": 0.41,
    "recall@5": 0.54,
    "recall@10": 0.67,
    "recall@20": 0.78,
    "recall@50": 0.91,
    "mrr": 0.4942,
    "avg_ms": 0.99,
    "total_misses": 33,
    "errors": {
      "miss_oov": 4,
      "miss_staple": 24,
      "miss_long_recipe": 1,
      "miss_overlap": 4
    }
  },
  {
    "approach": "tfidf-staple",
    "description": "TF-IDF produksi (max_features=1000, ngram 1-2) dengan pantry staple down-weighting",
    "queries": 100,
    "recall@1": 0.27,
    "recall@5": 0.49,
    "recall@10": 0.54,
    "recall@20": 0.69,
    "recall@50": 0.84,
    "mrr": 0.3747,
    "avg_ms": 1.0,
    "total_misses": 46,
    "errors": {
      "miss_oov": 6,
      "miss_staple": 32,
      "miss_long_recipe": 2,
      "miss_overlap": 6
    }
  },
  {
    "approach": "tfidf-unigram",
    "description": "TF-IDF unigram saja 

[info] Indexed 9,997 recipes with TF-IDF (0.65s)


[info] Retrieved Top-50 candidates for 60 queries; estimated calories for 1568/1570 unique candidate recipes (99.9%) in 0.68s



=== Pipeline 3 Evaluation & Ablation Study (PRD §§33-35) ===
Corpus: recipes_sample.csv | Queries: annotated_queries.json (60) | Top-N=50 -> Top-K=3 | Calorie mode: per_serving
Note: MRR measures first Top-K recipe with 0 constraint violations & weighted coverage >= 50%.

| Model | Weighted Cov@3 ↑ | Unweighted Cov@3 ↑ | Missing@3 ↓ | Non-Staple Miss@3 ↓ | Calorie Dist (kcal) ↓ | Constraint Viol ↓ | Excl Viol ↓ | Cal Viol ↓ | MRR ↑ | Fallback |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Model A: TF-IDF Only (Baseline) | 49.7% | 45.1% | 2.73 | 1.96 | 399.6 | 35.9% | 14.8% | 51.7% | 0.539 | 0.0% |
| Model B: TF-IDF + Coverage | 73.6% | 61.7% | 1.68 | 0.78 | 163.8 | 25.6% | 12.3% | 36.7% | 0.831 | 0.0% |
| Model C: TF-IDF + Cov + Missing Pen. | 74.2% | 61.7% | 1.69 | 0.74 | 210.3 | 26.5% | 12.3% | 38.3% | 0.831 | 0.0% |
| Model D: Full Proposed (Weighted) | 70.2% | 57.6% | 2.02 | 0.95 | 80.9 | 0.0% | 0.0% | 0.0% | 0.900 | 0.0% |
| Model D (Unweighted 

## 7. Test Suite — `tests/`

Seluruh test regresi repositori, dijalankan langsung di notebook. Fixture pytest
(`vocab`, `tmp_path`) diganti variabel/`tempfile` biasa; `test_pipeline3` tetap
`unittest.TestCase` dan dijalankan lewat runner yang sama.


**`tests/test_preprocessing.py`**


In [54]:
def test_singularize_and_normalize_text() -> None:
    assert singularize("potatoes") == "potato"
    assert singularize("tomatoes") == "tomato"
    assert singularize("eggs") == "egg"
    assert singularize("berries") == "berry"
    # normalize_text is a pure query-time fold (no stemming); only
    # normalize_ingredient folds the head noun.
    assert normalize_text("Fresh Green Onions!") == "fresh green onions"
    assert normalize_ingredient("Fresh Green Onions!") == "fresh green onion"

def test_ingredient_normalizer_aliases_and_qualifiers() -> None:
    normalizer = IngredientNormalizer()
    assert normalizer.resolve("scallions") == "green onion"
    assert normalizer.resolve("catsup") == "ketchup"
    assert normalizer.resolve("granulated sugar") == "sugar"
    assert normalizer.resolve("unsalted butter") == "butter"
    assert normalizer.extract_canonical_ingredients(
        ["chickens", "potatoes", "scallions", "green onions", "salt"]
    ) == ["chicken", "potato", "green onion", "salt"]

def test_parse_ingredient_quantities_units_and_containers() -> None:
    p1 = parse_ingredient("2 tbsp butter")
    assert p1["quantity"] == 2.0
    assert p1["unit"] == "tbsp"
    assert p1["ingredient"] == "butter"

    p2 = parse_ingredient("1 1/2 c. sugar")
    assert p2["quantity"] == 1.5
    assert p2["unit"] == "cup"
    assert p2["ingredient"] == "sugar"

    p3 = parse_ingredient("1 (16 oz.) can crushed pineapple, drained")
    assert p3["quantity"] == 16.0
    assert p3["unit"] == "oz"
    assert p3["ingredient"] == "pineapple"

    p4 = parse_ingredient("200 g chicken breast")
    assert p4["quantity"] == 200.0
    assert p4["unit"] == "g"
    assert p4["ingredient"] == "chicken breast"

    p5 = parse_ingredient("salt and pepper")
    assert p5["quantity"] is None
    assert p5["unit"] is None
    assert p5["ingredient"] == "salt and pepper"

    parsed_many = parse_ingredients(["1/2 cup milk", "3 medium apples, chopped"])
    assert len(parsed_many) == 2
    assert parsed_many[0]["quantity"] == 0.5
    assert parsed_many[1]["unit"] == "medium"

def test_calorie_estimator_estimation() -> None:
    mapper = RecipeCalorieEstimator()
    butter_lookup = mapper.lookup_calorie_per_100g("unsalted butter")
    assert butter_lookup is not None
    assert butter_lookup["canonical"] == "butter"
    assert float(butter_lookup["kcal_per_100g"]) > 500.0

    water_lookup = mapper.lookup_calorie_per_100g("cold water")
    assert water_lookup is not None
    assert water_lookup["match_type"] == "zero"
    assert float(water_lookup["kcal_per_100g"]) == 0.0

    recipe_est = mapper.estimate_recipe_calories(
        [
            "2 tbsp butter",
            "1/2 cup milk",
            "200 g chicken breast",
            "1 tsp salt",
        ],
        ner_ingredients=["butter", "milk", "chicken breast", "salt"],
    )
    assert float(recipe_est["total_calories"]) > 300.0
    assert float(recipe_est["calorie_coverage"]) == 1.0

def test_build_processed_corpus_subset():
    tmp_path = Path(tempfile.mkdtemp())
    out_parquet = tmp_path / "recipes.parquet"
    out_vocab = tmp_path / "ingredient_vocabulary.pkl"
    df = build_processed_corpus(
        corpus_path=SAMPLE_CSV,
        output_path=out_parquet,
        vocab_path=out_vocab,
        limit=25,
    )
    assert len(df) == 25
    assert out_parquet.exists()
    assert out_vocab.exists()
    assert {
        "recipe_id",
        "title",
        "ingredients",
        "canonical_ingredients",
        "canonical_text",
        "directions",
        "total_calories",
        "calorie_coverage",
    }.issubset(df.columns)
    vocab = IngredientNormalizer.load_vocabulary(out_vocab)
    assert len(vocab) > 100


**`tests/test_pipeline1.py`**

Pipeline 1 regression guards for behaviour the annotated set only implies.


In [55]:
def test_exclusion_clause_removes_earlier_availability():
    result = parse_query("I have potatoes, butter and garlic. Without milk or butter.")
    assert result["available_ingredients"] == ["potato", "garlic"]
    assert result["excluded_ingredients"] == ["milk", "butter"]

def test_upper_bound_is_not_an_exclusion():
    result = parse_query("I have beef and noodles. No more than 700 calories.")
    assert result["excluded_ingredients"] == []
    assert result["calorie_max"] == 700
    assert result["available_ingredients"] == ["beef", "noodle"]

def test_bare_kcal_number_is_not_a_constraint():
    result = parse_query("I have chicken and rice. 450 kcal.")
    assert result["calorie_target"] is None
    assert result["calorie_max"] is None
    assert result["calorie_min"] is None

def test_range_sets_both_bounds():
    result = parse_query("I have chicken, potato, carrot and onion. Between 400 and 600 calories.")
    assert (result["calorie_min"], result["calorie_max"], result["calorie_target"]) == (400, 600, None)

def test_longest_match_wins_over_substring():
    result = parse_query("Available: tomatoes, onions, garlic, olive oil.")
    assert result["available_ingredients"] == ["tomato", "onion", "garlic", "olive oil"]

def test_plural_exclusion_and_generic_nut():
    result = parse_query("I have oats, banana, milk and honey. Around 350 calories, no nuts.")
    assert result["excluded_ingredients"] == ["nut"]
    assert result["calorie_target"] == 350

def test_tilde_approx_target_is_recognized():
    result = parse_query("i have chicken potato carrot onion garlic. ~500 calories")
    assert result["available_ingredients"] == ["chicken", "potato", "carrot", "onion", "garlic"]
    assert result["calorie_target"] == 500

def test_two_exclusion_cues_in_one_clause():
    result = parse_query("I have salmon and rice. No butter and no cream.")
    assert result["excluded_ingredients"] == ["butter", "cream"]

def test_query_without_pantry_still_parses():
    result = parse_query("I want a low calorie dinner.")
    assert result == {
        "available_ingredients": [],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
 
def test_range_accepts_collapsed_symbol_separator():
    result = parse_query("I have rice. Between 300 & 500 cal.")
    assert (result["calorie_min"], result["calorie_max"]) == (300, 500)

def test_joined_compound_ingredient():
    result = parse_query("I have sweetpotato and spinach.")
    assert result["available_ingredients"] == ["sweet potato", "spinach"]


**`tests/test_approaches.py`**

Approach registry, trace runner, and retrieval approach guards.


In [56]:
def test_registry_lists_all_lexical_approaches():
    names = [entry["name"] for entry in available_approaches()]
    assert names[:4] == ["rule", "surface-exact", "difflib-0.82", "char-ngram-0.60"]
    # ner is registry-listed regardless of artifact presence (flagged via `available`)
    assert "ner" in names

def test_rule_approach_matches_parse_query_default():
    approach = get_approach("rule")
    for query in (
        "I have chicken, potato, carrot, onion and garlic. Around 500 calories and no cheese.",
        "no milk or butter, just beef",
        "sweetpotato and green onions",
    ):
        assert parse_query(query, vocab, approach.extract) == parse_query(query, vocab)

def test_trace_parse_query_records_spans():
    trace = trace_parse_query("I have eggs. Around 500 calories, no cheese.", get_approach("rule"), vocab)
    assert trace.structured_query == parse_query(
        "I have eggs. Around 500 calories, no cheese.", vocab
    )
    assert trace.calorie_spans, "calorie span must be recorded"
    assert trace.exclusion_spans, "exclusion span must be recorded"
    assert [m.canonical for m in trace.available_trace.matches] == ["egg"]
    assert trace.excluded_trace.ingredients == ["cheese"]

def test_fuzzy_approach_rescues_typos():
    approach = get_approach("difflib-0.82")
    trace = trace_parse_query("I have chiken and tommato.", approach, vocab)
    assert trace.structured_query["available_ingredients"] == ["chicken", "tomato"]
    methods = {m.method for m in trace.available_trace.matches}
    assert "fuzzy-difflib" in methods

def test_ner_approach_requires_artifacts():
    tmp_path = Path(tempfile.mkdtemp())
    missing = NerApproach(model_dir=tmp_path / "nope")
    if not missing.available:
        with pytest.raises(FileNotFoundError):
            missing._load()
    else:  # artifacts present in this checkout — loading must succeed
        missing._load()

def test_p2_approaches_share_candidate_schema():

    approach = TfidfApproach()
    candidates = approach.retrieve(
        {"available_ingredients": ["chicken", "potato"], "excluded_ingredients": []}, top_n=5
    )
    required = {"recipe_id", "title", "canonical_ingredients", "tfidf_score", "total_calories", "calorie_coverage"}
    assert candidates and required.issubset(candidates[0].keys())
    # harness compatibility (evaluate_retriever reads these directly)
    assert approach.recipe_ids is not None
    indices, scores = approach.retrieve_indices(
        {"available_ingredients": ["chicken"], "excluded_ingredients": []}, top_n=3
    )
    assert len(indices) == len(scores) == 3


**`tests/test_pipeline2.py`**


In [57]:
DUMMY_STRUCTURED_QUERIES = [
    {
        "available_ingredients": ["chicken", "potato", "carrot", "onion", "garlic"],
        "excluded_ingredients": ["cheese"],
        "calorie_target": 500,
        "calorie_min": None,
        "calorie_max": None,
    },
    {
        "available_ingredients": ["egg", "milk", "butter", "flour", "sugar"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": 600,
    },
    {
        "available_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
        "excluded_ingredients": ["milk"],
        "calorie_target": None,
        "calorie_min": 400,
        "calorie_max": 700,
    },
]

def _make_dummy_corpus() -> pd.DataFrame:
    rows = [
        {
            "recipe_id": 0,
            "title": "Chicken Potato Soup",
            "ingredients": ["200 g chicken", "2 potatoes", "1 carrot", "1 onion", "2 cloves garlic", "1 stalk celery"],
            "canonical_ingredients": ["chicken", "potato", "carrot", "onion", "garlic", "celery"],
            "canonical_text": "chicken potato carrot onion garlic celery",
            "directions": ["Boil all ingredients until tender."],
            "total_calories": 482.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 1,
            "title": "Basic Pancakes",
            "ingredients": ["1 cup flour", "1 cup milk", "1 egg", "2 tbsp butter", "2 tbsp sugar", "1 tsp salt"],
            "canonical_ingredients": ["flour", "milk", "egg", "butter", "sugar", "salt"],
            "canonical_text": "flour milk egg butter sugar salt",
            "directions": ["Mix batter and cook on griddle."],
            "total_calories": 520.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 2,
            "title": "Beef Tomato Pasta",
            "ingredients": ["250 g ground beef", "2 tomatoes", "1 onion", "2 cloves garlic", "200 g pasta"],
            "canonical_ingredients": ["ground beef", "tomato", "onion", "garlic", "pasta"],
            "canonical_text": "ground beef tomato onion garlic pasta",
            "directions": ["Brown beef, simmer with tomato, toss with pasta."],
            "total_calories": 640.0,
            "calorie_coverage": 1.0,
        },
        {
            "recipe_id": 3,
            "title": "Salted Water Broth",
            "ingredients": ["4 cups water", "2 tsp salt", "1 tsp pepper"],
            "canonical_ingredients": ["water", "salt", "pepper"],
            "canonical_text": "water salt pepper",
            "directions": ["Boil water with salt and pepper."],
            "total_calories": 0.0,
            "calorie_coverage": 1.0,
        },
    ]
    return pd.DataFrame(rows)

def test_dummy_query_vectorization_and_retrieval() -> None:
    corpus = _make_dummy_corpus()
    vectorizer = RecipeVectorizer(max_features=1000, ngram_range=(1, 2), use_staple_downweight=True)
    recipe_vectors = vectorizer.fit_transform(corpus["canonical_text"].tolist())
    retriever = CandidateRetriever(vectorizer=vectorizer, recipe_vectors=recipe_vectors, corpus=corpus)

    q_vec = vectorizer.transform_query(DUMMY_STRUCTURED_QUERIES[0])
    assert q_vec.shape == (1, recipe_vectors.shape[1])
    assert q_vec.nnz > 0

    candidates_soup = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[0], top_n=3)
    assert len(candidates_soup) == 3
    assert candidates_soup[0]["title"] == "Chicken Potato Soup"
    assert candidates_soup[0]["tfidf_score"] >= candidates_soup[1]["tfidf_score"]

    candidates_pancake = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[1], top_n=3)
    assert candidates_pancake[0]["title"] == "Basic Pancakes"

    candidates_pasta = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[2], top_n=3)
    assert candidates_pasta[0]["title"] == "Beef Tomato Pasta"

def test_staple_downweight_reduces_staple_dominance() -> None:
    corpus = _make_dummy_corpus()
    vec_weighted = RecipeVectorizer(max_features=1000, ngram_range=(1, 2), use_staple_downweight=True)
    mat_weighted = vec_weighted.fit_transform(corpus["canonical_text"].tolist())
    retriever = CandidateRetriever(vectorizer=vec_weighted, recipe_vectors=mat_weighted, corpus=corpus)

    dummy_query = {
        "available_ingredients": ["flour", "egg", "salt", "water", "pepper"],
        "excluded_ingredients": [],
        "calorie_target": None,
        "calorie_min": None,
        "calorie_max": None,
    }
    candidates = retriever.retrieve(dummy_query, top_n=2)
    assert candidates[0]["title"] == "Basic Pancakes"

def test_pipeline2_end_to_end_with_artifacts_and_evaluation():
    tmp_path = Path(tempfile.mkdtemp())
    corpus_parquet = tmp_path / "recipes.parquet"
    vocab_pkl = tmp_path / "ingredient_vocabulary.pkl"
    vec_pkl = tmp_path / "tfidf_vectorizer.pkl"
    mat_pkl = tmp_path / "recipe_vectors.pkl"

    build_processed_corpus(
        corpus_path=SAMPLE_CSV,
        output_path=corpus_parquet,
        vocab_path=vocab_pkl,
        limit=100,
    )
    fit_corpus_tfidf(
        corpus_parquet=corpus_parquet,
        vectorizer_path=vec_pkl,
        vectors_path=mat_pkl,
        max_features=1000,
        use_staple_downweight=True,
    )

    retriever = CandidateRetriever.from_artifacts(
        corpus_path=corpus_parquet,
        vectorizer_path=vec_pkl,
        vectors_path=mat_pkl,
    )
    candidates = retriever.retrieve(DUMMY_STRUCTURED_QUERIES[1], top_n=10)
    assert len(candidates) == 10
    assert all("tfidf_score" in c and "canonical_ingredients" in c for c in candidates)

    queries = generate_in_memory_queries(retriever.corpus, num_queries=20, seed=42)
    metrics = evaluate_retriever(retriever, queries, ks=(1, 5, 10, 50))
    assert metrics["evaluated_queries"] == 20.0
    assert 0.0 <= metrics["recall@50"] <= 1.0
    assert 0.0 <= metrics["mrr"] <= 1.0


In [ ]:
# Source CandidateRetriever now requires the per-serving calorie column;
# the inline dummy corpus predates it. Extend the snapshot in place.
_original_make_dummy_corpus = _make_dummy_corpus
def _make_dummy_corpus():
    frame = _original_make_dummy_corpus()
    if "calories_per_serving" not in frame.columns:
        frame["calories_per_serving"] = frame["total_calories"]
    return frame

**`tests/test_pipeline3.py`**

Unit and integration tests for Pipeline 3 (Pantry-Aware Multi-Criteria Reranking).  Compatible with both ``pytest`` and standard-library ``python -m unittest``.


In [ ]:
import pytest
_p3_exit = pytest.main(["-q", str(ROOT / "tests" / "test_pipeline3.py")])
assert _p3_exit == 0


In [59]:
vocab = load_vocabulary()    # pengganti fixture pytest scope-module

In [ ]:
import pytest
_suite_exit = pytest.main(["-q", str(ROOT / "tests")])
assert _suite_exit == 0


## 8. Demo End-to-End

Pantry bebas -> Pipeline 1 (parse + trace) -> Pipeline 2 (retrieval TF-IDF; dense bila
tersedia) -> Pipeline 3 (rerank multi-kriteria) -> kartu rekomendasi.


In [61]:
demo_query = "I have chicken, rice, onion, and garlic. Around 600 calories and no peanuts."

# Pipeline 1: parse + trace
trace = trace_parse_query(demo_query, get_approach("difflib-0.82"), vocab)
sq = trace.structured_query
print("structured query:", sq)
print("matched :", [(m.surface, m.canonical, m.method) for m in trace.available_trace.matches])
print("excluded:", trace.excluded_trace.ingredients)

# Pipeline 2: retrieval kandidat
tfidf_approach = TfidfApproach()
candidates = tfidf_approach.retrieve(sq, top_n=50)
print(f"\nkandidat TF-IDF: {len(candidates)} | teratas: {candidates[0]['title']!r}")

# Pipeline 3: rerank multi-kriteria -> kartu rekomendasi
result = recommend(candidates, sq, top_k=3, processed_dir=DEFAULT_PROCESSED_DIR)
print(format_recommendations(result))

structured query: {'available_ingredients': ['chicken', 'rice', 'onion', 'garlic'], 'excluded_ingredients': ['peanut'], 'calorie_target': 600, 'calorie_min': None, 'calorie_max': None}
matched : [('chicken', 'chicken', 'vocabulary'), ('rice', 'rice', 'vocabulary'), ('onion', 'onion', 'vocabulary'), ('garlic', 'garlic', 'vocabulary')]
excluded: ['peanut']

kandidat TF-IDF: 50 | teratas: 'Shrimp And Scallop Pilaf'


#1 Individual roast chicken leg quarters
Overall Match       : 75% (score=0.638)
Ingredient Coverage : 100% (1/1 ingredients)
Estimated Calories  : 619 kcal
Calorie Match       : 97%

YOU HAVE:
  ✓ chicken

MISSING:
  ✓ nothing missing!

CONSTRAINTS:
  ✓ approximately 600 kcal (est. 619 kcal)
  ✓ no peanut detected

INGREDIENTS:
  - 8 chicken leg quarters

DIRECTIONS:
  1. Season your chicken and place in pan
  2. Place in preheated oven on 350.
  3. Cook uncovered for 2 hours or until done and crispy.

------------------------------------------------------------

#2 Rotisser

In [62]:
# Perbandingan retrieval TF-IDF vs dense (bila model sentence-transformers tersedia)
try:
    dense_approach = get_retrieval_approach("dense")
    dense_candidates = dense_approach.retrieve(sq, top_n=50)
    result_dense = recommend(dense_candidates, sq, top_k=3, processed_dir=DEFAULT_PROCESSED_DIR)
    print("=== Dense retrieval (top-3) ===")
    print("\n".join(format_recommendations(result_dense).splitlines()[:12]))
    print("\n=== TF-IDF retrieval (top-3) ===")
    print("\n".join(format_recommendations(result).splitlines()[:12]))
except Exception as exc:  # model/language hub tidak tersedia
    print(f"dense approach dilewati: {exc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15047.49it/s]

=== Dense retrieval (top-3) ===


#1 Adobo
Overall Match       : 66% (score=0.557)
Ingredient Coverage : 72% (2/5 ingredients)
Estimated Calories  : 330 kcal
Calorie Match       : 55%

YOU HAVE:
  ✓ garlic
  ✓ onion


=== TF-IDF retrieval (top-3) ===


#1 Individual roast chicken leg quarters
Overall Match       : 75% (score=0.638)
Ingredient Coverage : 100% (1/1 ingredients)
Estimated Calories  : 619 kcal
Calorie Match       : 97%

YOU HAVE:
  ✓ chicken

MISSING:
